In [12]:
#georeferencing, layerstacking and rotating
#working

import os
import csv
import glob
import numpy as np
import rasterio
from rasterio.transform import Affine
from rasterio.crs import CRS
from exiftool import ExifTool
import cv2

# 📂 Set folder paths
input_folder = r"C:/Tarun/CIFRI_ponds/0002_MX/"
output_folder = r"C:/Tarun/CIFRI_ponds/0002_MX/test"
csv_metadata_file = r"C:/Tarun/CIFRI_ponds/0002_MX/metadata.csv"
os.makedirs(output_folder, exist_ok=True)

# 📌 Define band order
band_order = ["GRE", "RED", "REG", "NIR"]

def clip_valid_area(stacked_image):
    """
    Clips areas where at least one band has a zero value.
    """
    # Create a mask where all bands have nonzero values
    valid_mask = np.all(stacked_image > 0, axis=0)

    # Find nonzero regions (bounding box)
    coords = np.argwhere(valid_mask)
    if coords.size == 0:
        print("⚠️ No valid area found after alignment. Skipping clipping.")
        return stacked_image  # Return original if no valid area found

    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0) + 1  # Add 1 to include the last pixel

    # Crop the image using the bounding box
    clipped_image = stacked_image[:, y_min:y_max, x_min:x_max]
    print(f"✅ Clipped image to size: {clipped_image.shape}")

    return clipped_image

# 📌 Function to align images
def align_images(reference, image):
    sift = cv2.SIFT_create()

    # Convert images to uint8
    reference = cv2.convertScaleAbs(reference, alpha=(255.0 / reference.max())) if reference.dtype != np.uint8 else reference
    image = cv2.convertScaleAbs(image, alpha=(255.0 / image.max())) if image.dtype != np.uint8 else image

    keypoints1, descriptors1 = sift.detectAndCompute(reference, None)
    keypoints2, descriptors2 = sift.detectAndCompute(image, None)

    if descriptors1 is None or descriptors2 is None or len(descriptors1) < 2 or len(descriptors2) < 2:
        print("⚠️ Insufficient features for matching. Skipping alignment.")
        return image
    
    index_params = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(descriptors1, descriptors2, k=2)

    good_matches = [m for m, n in matches if m.distance < 0.7 * n.distance]
    
    if len(good_matches) < 10:
        print("⚠️ Not enough matches for alignment. Using original image.")
        return image
    
    src_pts = np.float32([keypoints2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([keypoints1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    H, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    if H is None:
        print("⚠️ Homography computation failed. Using original image.")
        return image

    aligned = cv2.warpPerspective(image, H, (reference.shape[1], reference.shape[0]))
    return aligned

# 📌 Function to rotate an image
def ModifiedWay(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    angle = -angle
    
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    abs_cos = abs(rotation_matrix[0, 0])
    abs_sin = abs(rotation_matrix[0, 1])
    
    new_w = int(h * abs_sin + w * abs_cos)
    new_h = int(h * abs_cos + w * abs_sin)
    
    rotation_matrix[0, 2] += (new_w / 2) - center[0]
    rotation_matrix[1, 2] += (new_h / 2) - center[1]
    
    rotated_image = cv2.warpAffine(image, rotation_matrix, (new_w, new_h), 
                                   flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    return rotated_image

# 📌 Function to process images
def process_images():
    image_metadata = {}
    
    with open(csv_metadata_file, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            image_metadata[row["Filename"]] = {
                "Yaw": float(row["Yaw"]),
                "Latitude": float(row["Latitude"]),
                "Longitude": float(row["Longitude"])
            }
    
    image_groups = {}
    for band in band_order:
        band_images = sorted(glob.glob(os.path.join(input_folder, f"*_{band}.tif")))
        for img in band_images:
            capture_id = "_".join(img.split("_")[:-1])
            if capture_id not in image_groups:
                image_groups[capture_id] = {}
            image_groups[capture_id][band] = img
    
    for capture_id, images in image_groups.items():
        if len(images) != 4:
            print(f"⚠️ Skipping {capture_id}, missing bands.")
            continue
        
        base_img = os.path.basename(images["GRE"])
        if base_img not in image_metadata:
            print(f"⚠️ Metadata missing for {base_img}")
            continue
        
        yaw = image_metadata[base_img]["Yaw"]
        lat = image_metadata[base_img]["Latitude"]
        lon = image_metadata[base_img]["Longitude"]
        
        img_data = {}
        for band, path in images.items():
            with rasterio.open(path) as src:
                image = src.read(1)
                rotated_img = ModifiedWay(image, yaw)
                img_data[band] = rotated_img
        
        # Align RED, REG, and NIR bands to GRE
        img_data["RED"] = align_images(img_data["GRE"], img_data["RED"])
        img_data["REG"] = align_images(img_data["GRE"], img_data["REG"])
        img_data["NIR"] = align_images(img_data["GRE"], img_data["NIR"])
        
        stacked_image = np.stack([img_data["GRE"], img_data["RED"], img_data["REG"], img_data["NIR"]], axis=0)
        
        pixel_size = 0.023 / 111320
        center_x = lon - (pixel_size * stacked_image.shape[2] / 2)
        center_y = lat + (pixel_size * stacked_image.shape[1] / 2)
        
        new_transform = Affine.translation(center_x, center_y) * Affine.scale(pixel_size, -pixel_size)
        output_path = os.path.join(output_folder, f"{os.path.basename(capture_id)}_rotated_stacked.tif")
        # Define WGS 84 manually using WKT
        wgs84_crs = CRS.from_wkt(
            'GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],'
            'PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433]]'
        )

        with rasterio.open(
            output_path, "w",
            driver="GTiff",
            height=stacked_image.shape[1],
            width=stacked_image.shape[2],
            count=4,
            dtype=stacked_image.dtype,
            crs=wgs84_crs,
            transform=new_transform
        ) as dst:
            for i in range(4):
                dst.write(stacked_image[i], i + 1)

        print(f"✅ Rotated, aligned, and georeferenced: {output_path}")

# Run the process
process_images()
print("🎯 Processing complete!")


✅ Rotated, aligned, and georeferenced: C:/Tarun/CIFRI_ponds/0002_MX/test\IMG_241207_080051_0000_rotated_stacked.tif
✅ Rotated, aligned, and georeferenced: C:/Tarun/CIFRI_ponds/0002_MX/test\IMG_241207_080059_0001_rotated_stacked.tif
✅ Rotated, aligned, and georeferenced: C:/Tarun/CIFRI_ponds/0002_MX/test\IMG_241207_080110_0002_rotated_stacked.tif


KeyboardInterrupt: 

In [ ]:
#layerstacking, rotating, clipping, georeferencing
#working
import os
import csv
import glob
import numpy as np
import rasterio
from rasterio.transform import Affine
from rasterio.crs import CRS
from exiftool import ExifTool
import cv2

# 📂 Set folder paths
input_folder = r"C:/Tarun/CIFRI_ponds/0002_MX/"
output_folder = r"C:/Tarun/CIFRI_ponds/0002_MX/test"
csv_metadata_file = r"C:/Tarun/CIFRI_ponds/0002_MX/metadata.csv"
os.makedirs(output_folder, exist_ok=True)

# 📌 Define band order
band_order = ["GRE", "RED", "REG", "NIR"]

# 📌 Function to clip layer-stacked image
def clip_layerstacked_image(stacked_image):
    """
    Clip the layer-stacked image such that every pixel has 4 non-zero values.
    If any band has a zero value for a pixel, set all bands for that pixel to zero.
    """
    # Create a mask where any band has a zero value
    mask = (stacked_image == 0).any(axis=0)
    
    # Apply the mask to all bands
    clipped_image = np.where(mask, 0, stacked_image)
    
    return clipped_image

# 📌 Function to align images
def align_images(reference, image):
    sift = cv2.SIFT_create()

    # Convert images to uint8
    reference = cv2.convertScaleAbs(reference, alpha=(255.0 / reference.max())) if reference.dtype != np.uint8 else reference
    image = cv2.convertScaleAbs(image, alpha=(255.0 / image.max())) if image.dtype != np.uint8 else image

    keypoints1, descriptors1 = sift.detectAndCompute(reference, None)
    keypoints2, descriptors2 = sift.detectAndCompute(image, None)

    if descriptors1 is None or descriptors2 is None or len(descriptors1) < 2 or len(descriptors2) < 2:
        print("⚠️ Insufficient features for matching. Skipping alignment.")
        return image
    
    index_params = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(descriptors1, descriptors2, k=2)

    good_matches = [m for m, n in matches if m.distance < 0.7 * n.distance]
    
    if len(good_matches) < 10:
        print("⚠️ Not enough matches for alignment. Using original image.")
        return image
    
    src_pts = np.float32([keypoints2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([keypoints1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    H, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    if H is None:
        print("⚠️ Homography computation failed. Using original image.")
        return image

    aligned = cv2.warpPerspective(image, H, (reference.shape[1], reference.shape[0]))
    return aligned

# 📌 Function to rotate an image
def ModifiedWay(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    angle = -angle
    
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    abs_cos = abs(rotation_matrix[0, 0])
    abs_sin = abs(rotation_matrix[0, 1])
    
    new_w = int(h * abs_sin + w * abs_cos)
    new_h = int(h * abs_cos + w * abs_sin)
    
    rotation_matrix[0, 2] += (new_w / 2) - center[0]
    rotation_matrix[1, 2] += (new_h / 2) - center[1]
    
    rotated_image = cv2.warpAffine(image, rotation_matrix, (new_w, new_h), 
                                   flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    return rotated_image

# 📌 Function to process images
def process_images():
    image_metadata = {}
    
    with open(csv_metadata_file, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            image_metadata[row["Filename"]] = {
                "Yaw": float(row["Yaw"]),
                "Latitude": float(row["Latitude"]),
                "Longitude": float(row["Longitude"])
            }
    
    image_groups = {}
    for band in band_order:
        band_images = sorted(glob.glob(os.path.join(input_folder, f"*_{band}.tif")))
        for img in band_images:
            capture_id = "_".join(img.split("_")[:-1])
            if capture_id not in image_groups:
                image_groups[capture_id] = {}
            image_groups[capture_id][band] = img
    
    for capture_id, images in image_groups.items():
        if len(images) != 4:
            print(f"⚠️ Skipping {capture_id}, missing bands.")
            continue
        
        base_img = os.path.basename(images["GRE"])
        if base_img not in image_metadata:
            print(f"⚠️ Metadata missing for {base_img}")
            continue
        
        yaw = image_metadata[base_img]["Yaw"]
        lat = image_metadata[base_img]["Latitude"]
        lon = image_metadata[base_img]["Longitude"]
        
        img_data = {}
        for band, path in images.items():
            with rasterio.open(path) as src:
                image = src.read(1)
                rotated_img = ModifiedWay(image, yaw)
                img_data[band] = rotated_img
        
        # Align RED, REG, and NIR bands to GRE
        img_data["RED"] = align_images(img_data["GRE"], img_data["RED"])
        img_data["REG"] = align_images(img_data["GRE"], img_data["REG"])
        img_data["NIR"] = align_images(img_data["GRE"], img_data["NIR"])
        
        # Stack the images
        stacked_image = np.stack([img_data["GRE"], img_data["RED"], img_data["REG"], img_data["NIR"]], axis=0)
        
        # Clip the stacked image to remove invalid areas
        clipped_image = clip_layerstacked_image(stacked_image)
        
        # Calculate pixel size in degrees (assuming GSD is 0.023 meters/pixel)
        pixel_size = 0.023 / 111320  # Convert meters to degrees (1 degree ≈ 111,320 meters)
        
        # Calculate the center coordinates of the clipped image
        center_x = lon - (pixel_size * clipped_image.shape[2] / 2)
        center_y = lat + (pixel_size * clipped_image.shape[1] / 2)
        
        # Create the geotransform
        new_transform = Affine.translation(center_x, center_y) * Affine.scale(pixel_size, -pixel_size)
        
        # Define WGS 84 manually using WKT
        wgs84_crs = CRS.from_wkt(
            'GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],'
            'PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433]]'
        )

        # Write the georeferenced image to a GeoTIFF file
        output_path = os.path.join(output_folder, f"{os.path.basename(capture_id)}_rotated_stacked_clipped.tif")
        with rasterio.open(
            output_path, "w",
            driver="GTiff",
            height=clipped_image.shape[1],
            width=clipped_image.shape[2],
            count=4,
            dtype=clipped_image.dtype,
            crs=wgs84_crs,
            transform=new_transform
        ) as dst:
            for i in range(4):
                dst.write(clipped_image[i], i + 1)

        print(f"✅ Rotated, aligned, clipped, and georeferenced: {output_path}")

# Run the process
process_images()
print("🎯 Processing complete!")

✅ Rotated, aligned, clipped, and georeferenced: C:/Tarun/CIFRI_ponds/0002_MX/test\IMG_241207_080051_0000_rotated_stacked_clipped.tif
✅ Rotated, aligned, clipped, and georeferenced: C:/Tarun/CIFRI_ponds/0002_MX/test\IMG_241207_080059_0001_rotated_stacked_clipped.tif
✅ Rotated, aligned, clipped, and georeferenced: C:/Tarun/CIFRI_ponds/0002_MX/test\IMG_241207_080110_0002_rotated_stacked_clipped.tif
✅ Rotated, aligned, clipped, and georeferenced: C:/Tarun/CIFRI_ponds/0002_MX/test\IMG_241207_080124_0003_rotated_stacked_clipped.tif
✅ Rotated, aligned, clipped, and georeferenced: C:/Tarun/CIFRI_ponds/0002_MX/test\IMG_241207_080128_0004_rotated_stacked_clipped.tif
✅ Rotated, aligned, clipped, and georeferenced: C:/Tarun/CIFRI_ponds/0002_MX/test\IMG_241207_080130_0005_rotated_stacked_clipped.tif
✅ Rotated, aligned, clipped, and georeferenced: C:/Tarun/CIFRI_ponds/0002_MX/test\IMG_241207_080132_0006_rotated_stacked_clipped.tif
✅ Rotated, aligned, clipped, and georeferenced: C:/Tarun/CIFRI_ponds/

In [ ]:
#clipping georeferencing, layerstacking and rotating
#working

import os
import csv
import glob
import numpy as np
import rasterio
from rasterio.transform import Affine
from rasterio.crs import CRS
from exiftool import ExifTool
import cv2

# 📂 Set folder paths
input_folder = r"C:/Tarun/CIFRI_ponds/0002_MX/"
output_folder = r"C:/Tarun/CIFRI_ponds/0002_MX/test"
csv_metadata_file = r"C:/Tarun/CIFRI_ponds/0002_MX/metadata.csv"
os.makedirs(output_folder, exist_ok=True)

# 📌 Define band order
band_order = ["GRE", "RED", "REG", "NIR"]

# 📌 Extract metadata and save to CSV
def extract_metadata():
    metadata_list = []
    image_paths = glob.glob(os.path.join(input_folder, "*.tif"))
    
    with ExifTool() as et:
        for img_path in image_paths:
            metadata = et.get_metadata(img_path)
            yaw = metadata.get("XMP:Yaw")
            pitch = metadata.get("XMP:Pitch")
            roll = metadata.get("XMP:Roll")
            lat = metadata.get("EXIF:GPSLatitude")
            lon = metadata.get("EXIF:GPSLongitude")
            
            if None in (yaw, pitch, roll, lat, lon):
                print(f"⚠️ Missing metadata for {img_path}")
                continue
            
            metadata_list.append([os.path.basename(img_path), float(yaw), float(pitch), float(roll), float(lat), float(lon)])
    
    with open(csv_metadata_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Filename", "Yaw", "Pitch", "Roll", "Latitude", "Longitude"])
        writer.writerows(metadata_list)
    
    print("✅ Metadata extracted and saved to CSV")

# 📌 Function to apply yaw correction only
def apply_yaw(image, yaw):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    
    # Convert yaw to radians
    yaw = np.radians(-yaw)
    
    # Create rotation matrix for yaw only
    M = cv2.getRotationMatrix2D(center, np.degrees(yaw), 1.0)
    
    # Calculate the new bounding box
    cos = np.abs(M[0, 0])
    sin = np.abs(M[0, 1])
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))
    
    # Adjust the transformation matrix to include translation
    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]
    
    # Apply the rotation with the new size
    rotated_image = cv2.warpAffine(image, M, (new_w, new_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    
    return rotated_image

def visualize_matches(reference, image, keypoints1, keypoints2, matches, output_path="matches.png"):
    # Draw matches
    matched_image = cv2.drawMatches(reference, keypoints1, image, keypoints2, matches, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    
    # Save the result to disk
    cv2.imwrite(output_path, matched_image)
    print(f"✅ Matches saved to {output_path}")

def clip_layerstacked_image(stacked_image):
    """
    Clip the layer-stacked image such that every pixel has 4 non-zero values.
    If any band has a zero value for a pixel, set all bands for that pixel to zero.
    """
    # Create a mask where any band has a zero value
    mask = (stacked_image == 0).any(axis=0)
    
    # Debugging output
    print(f"Mask shape: {mask.shape}, True values: {np.sum(mask)}")
    
    # Apply the mask to all bands
    clipped_image = np.where(mask, 0, stacked_image)
    
    return clipped_image

# 📌 Function to align images
def align_images(reference, image):
    sift = cv2.SIFT_create()

    # Convert images to uint8
    reference = cv2.convertScaleAbs(reference, alpha=(255.0 / reference.max())) if reference.dtype != np.uint8 else reference
    image = cv2.convertScaleAbs(image, alpha=(255.0 / image.max())) if image.dtype != np.uint8 else image

    keypoints1, descriptors1 = sift.detectAndCompute(reference, None)
    keypoints2, descriptors2 = sift.detectAndCompute(image, None)

    if descriptors1 is None or descriptors2 is None or len(descriptors1) < 2 or len(descriptors2) < 2:
        print("⚠️ Insufficient features for matching. Skipping alignment.")
        return image
    
    index_params = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(descriptors1, descriptors2, k=2)

    good_matches = [m for m, n in matches if m.distance < 0.7 * n.distance]
    
    if len(good_matches) < 10:
        print("⚠️ Not enough matches for alignment. Using original image.")
        return image
    
    src_pts = np.float32([keypoints2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([keypoints1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    H, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    if H is None:
        print("⚠️ Homography computation failed. Using original image.")
        return image

    aligned = cv2.warpPerspective(image, H, (reference.shape[1], reference.shape[0]))
    return aligned

# 📌 Process images
def process_images():
    with open(csv_metadata_file, "r") as f:
        reader = csv.DictReader(f)
        image_metadata = {row["Filename"]: {key: float(row[key]) for key in row if key != "Filename"} for row in reader}
    
    image_groups = {}
    for band in band_order:
        for img in sorted(glob.glob(os.path.join(input_folder, f"*_{band}.tif"))):
            capture_id = "_".join(img.split("_")[:-1])
            image_groups.setdefault(capture_id, {})[band] = img
    
    for capture_id, images in image_groups.items():
        if len(images) != 4:
            print(f"⚠️ Skipping {capture_id}, missing bands.")
            continue
        
        base_img = os.path.basename(images["GRE"])
        if base_img not in image_metadata:
            print(f"⚠️ Metadata missing for {base_img}")
            continue
        
        yaw, pitch, roll, lat, lon = image_metadata[base_img].values()
        img_data = {}
        for band, path in images.items():
            # Read the image
            with rasterio.open(path) as src:
                img = src.read(1)
                print(f"Original {band} image shape: {img.shape}, min: {img.min()}, max: {img.max()}")
            
            # Apply yaw correction only
            rotated_img = apply_yaw(img, yaw)
            print(f"Rotated {band} image shape: {rotated_img.shape}, min: {rotated_img.min()}, max: {rotated_img.max()}")
            
            # Store the rotated image
            img_data[band] = rotated_img
        
        # Align images (optional, if alignment is needed)
        reference = img_data["GRE"]  # Use GRE band as reference
        for band in band_order:
            if band != "GRE":
                img_data[band] = align_images(reference, img_data[band])
                print(f"Aligned {band} image shape: {img_data[band].shape}, min: {img_data[band].min()}, max: {img_data[band].max()}")
        
        # Stack the images
        stacked_image = np.stack([img_data[b] for b in band_order], axis=0)
        print(f"Stacked image shape: {stacked_image.shape}, min: {stacked_image.min()}, max: {stacked_image.max()}")
        
        # Clip the layer-stacked image
        clipped_image = clip_layerstacked_image(stacked_image)
        print(f"Clipped image shape: {clipped_image.shape}, min: {clipped_image.min()}, max: {clipped_image.max()}")
        
        # Write the clipped image to a GeoTIFF file
        wgs84_crs = CRS.from_wkt(
            'GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],'
            'PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433]]'
        )
        output_path = os.path.join(output_folder, f"{os.path.basename(capture_id)}_aligned_clipped.tif")
        with rasterio.open(output_path, "w", driver="GTiff", height=clipped_image.shape[1], width=clipped_image.shape[2], count=4, dtype=clipped_image.dtype, crs=wgs84_crs) as dst:
            for i in range(4):
                dst.write(clipped_image[i], i + 1)
        
        print(f"✅ Processed: {output_path}")  
        
extract_metadata()
process_images()
print("🎯 Processing complete!")

✅ Metadata extracted and saved to CSV
Original GRE image shape: (960, 1280), min: 6464, max: 65472
Rotated GRE image shape: (1599, 1519), min: 0, max: 65472
Original RED image shape: (960, 1280), min: 5440, max: 65472
Rotated RED image shape: (1599, 1519), min: 0, max: 65472
Original REG image shape: (960, 1280), min: 6272, max: 65472
Rotated REG image shape: (1599, 1519), min: 0, max: 65472
Original NIR image shape: (960, 1280), min: 6080, max: 65472
Rotated NIR image shape: (1599, 1519), min: 0, max: 65472
Aligned RED image shape: (1599, 1519), min: 0, max: 255
Aligned REG image shape: (1599, 1519), min: 0, max: 255
Aligned NIR image shape: (1599, 1519), min: 0, max: 255
Stacked image shape: (4, 1599, 1519), min: 0, max: 65472
Mask shape: (1599, 1519), True values: 1294764
Clipped image shape: (4, 1599, 1519), min: 0, max: 65472
✅ Processed: C:/Tarun/CIFRI_ponds/0002_MX/test\IMG_241207_080051_0000_aligned_clipped.tif
Original GRE image shape: (960, 1280), min: 6656, max: 65472
Rotate

In [25]:
#only georeferencing_working
import os
import math
from pathlib import Path
from osgeo import gdal, osr
from exiftool import ExifTool

def calculate_sensor_width(hfov, focal_length):
    """Calculate sensor width using HFOV and focal length."""
    return 2 * focal_length * math.tan(math.radians(hfov / 2))

def calculate_gsd(sensor_width, image_width, flight_altitude, focal_length):
    """Calculate Ground Sampling Distance (GSD)."""
    return (sensor_width * flight_altitude) / (image_width * focal_length)

def extract_metadata(image_path):
    """Extract required metadata from image using ExifTool."""
    with ExifTool() as et:
        metadata = et.get_metadata(str(image_path))
        #metadata = et.get_metadata(image_path)
        focal_length = float(metadata.get("EXIF:FocalLength", 3.98))  # Default to 3.98 mm if missing
        flight_altitude = float(metadata.get("XMP:RelativeAltitude", 50))  # Default to 50m if missing
        latitude = float(metadata.get("EXIF:GPSLatitude", 0))
        longitude = float(metadata.get("EXIF:GPSLongitude", 0))
        image_width = int(metadata.get("EXIF:ExifImageWidth", 1280))  # Default width if missing
        image_height = int(metadata.get("EXIF:ExifImageHeight", 960))  # Default height if missing
    
    return focal_length, flight_altitude, latitude, longitude, image_width, image_height

def georeference_image(image_path, output_path, hfov):
    """Georeference an image based on extracted metadata."""
    ds = gdal.Open(str(image_path))
    if ds is None:
        print(f"Error: Unable to open {image_path}")
        return
    
    # Extract metadata
    focal_length, flight_altitude, lat, lon, img_width, img_height = extract_metadata(image_path)
    
    # Calculate sensor width and GSD
    sensor_width = calculate_sensor_width(hfov, focal_length)
    gsd = calculate_gsd(sensor_width, img_width, flight_altitude, focal_length)
    
    # Convert pixel size to degrees
    pixel_size_lat = gsd / 111320  # Degrees latitude per meter
    pixel_size_long = gsd / (111320 * math.cos(math.radians(lat)))
    
    # Define geotransform
    center_x = lon - (pixel_size_long * ds.RasterXSize / 2)
    center_y = lat + (pixel_size_lat * ds.RasterYSize / 2)
    geotransform = (center_x, pixel_size_long, 0, center_y, 0, -pixel_size_lat)
    
    # Set up output file
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(str(output_path), ds.RasterXSize, ds.RasterYSize, ds.RasterCount, ds.GetRasterBand(1).DataType)
    out_ds.SetGeoTransform(geotransform)
    
    # Set spatial reference
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(4326)
    out_ds.SetProjection(srs.ExportToWkt())
    
    # Copy raster bands
    for i in range(1, ds.RasterCount + 1):
        out_ds.GetRasterBand(i).WriteArray(ds.GetRasterBand(i).ReadAsArray())
    
    out_ds = None
    ds = None
    print(f"Georeferenced image saved: {output_path}")

def process_images(input_folder, output_folder, hfov):
    """Process all images in a folder."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(".tif"):
            input_path = Path(input_folder) / filename
            output_path = Path(output_folder) / f"georeferenced_{filename}"
            georeference_image(input_path, output_path, hfov)

# Define folders
input_folder = r"C:\\Tarun\\CIFRI_ponds\\0002_MX"
output_folder = r"C:\\Tarun\\CIFRI_ponds\\0002_MX\\geo"
hfov = 64  # Horizontal Field of View in degrees

# Run processing
process_images(input_folder, output_folder, hfov)


Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080051_0000_GRE.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080051_0000_NIR.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080051_0000_RED.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080051_0000_REG.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080059_0001_GRE.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080059_0001_NIR.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080059_0001_RED.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080059_0001_REG.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080110_0002_GRE.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX

KeyboardInterrupt: 

In [1]:
#georeferencing with extra metadata
import os
import math
import subprocess
from pathlib import Path
from osgeo import gdal, osr
from exiftool import ExifTool

def calculate_sensor_width(hfov, focal_length):
    """Calculate sensor width using HFOV and focal length."""
    return 2 * focal_length * math.tan(math.radians(hfov / 2))

def calculate_gsd(sensor_width, image_width, flight_altitude, focal_length):
    """Calculate Ground Sampling Distance (GSD)."""
    return (sensor_width * flight_altitude) / (image_width * focal_length)

def extract_metadata(image_path):
    """Extract required metadata from image using ExifTool."""
    with ExifTool() as et:
        metadata = et.get_metadata(str(image_path))
        focal_length = float(metadata.get("EXIF:FocalLength", 3.98))  # Default to 3.98 mm if missing
        flight_altitude = float(metadata.get("XMP:RelativeAltitude", 50))  # Default to 50m if missing
        latitude = float(metadata.get("EXIF:GPSLatitude", 0))
        longitude = float(metadata.get("EXIF:GPSLongitude", 0))
        image_width = int(metadata.get("EXIF:ExifImageWidth", 1280))  # Default width if missing
        image_height = int(metadata.get("EXIF:ExifImageHeight", 960))  # Default height if missing
    
    return focal_length, flight_altitude, latitude, longitude, image_width, image_height

def georeference_image(image_path, output_path, hfov):
    """Georeference an image while retaining metadata."""
    ds = gdal.Open(str(image_path))
    if ds is None:
        print(f"Error: Unable to open {image_path}")
        return
    
    # Extract metadata
    focal_length, flight_altitude, lat, lon, img_width, img_height = extract_metadata(image_path)
    
    # Calculate sensor width and GSD
    sensor_width = calculate_sensor_width(hfov, focal_length)
    gsd = calculate_gsd(sensor_width, img_width, flight_altitude, focal_length)
    
    # Convert pixel size to degrees
    pixel_size_lat = gsd / 111320  # Degrees latitude per meter
    pixel_size_long = gsd / (111320 * math.cos(math.radians(lat)))
    
    # Define geotransform
    center_x = lon - (pixel_size_long * ds.RasterXSize / 2)
    center_y = lat + (pixel_size_lat * ds.RasterYSize / 2)
    geotransform = (center_x, pixel_size_long, 0, center_y, 0, -pixel_size_lat)
    
    # Set up output file
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(str(output_path), ds.RasterXSize, ds.RasterYSize, ds.RasterCount, ds.GetRasterBand(1).DataType)
    out_ds.SetGeoTransform(geotransform)
    
    # Set spatial reference
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(4326)
    out_ds.SetProjection(srs.ExportToWkt())
    
    # Copy raster bands
    for i in range(1, ds.RasterCount + 1):
        out_ds.GetRasterBand(i).WriteArray(ds.GetRasterBand(i).ReadAsArray())
    
    # ✅ Copy EXIF and XMP metadata
    out_ds.SetMetadata(ds.GetMetadata())  # Standard metadata
    xmp_metadata = ds.GetMetadata("xml:XMP")
    if xmp_metadata:
        out_ds.SetMetadata(xmp_metadata, "xml:XMP")  # XMP metadata

    out_ds = None
    ds = None
    print(f"Georeferenced image saved: {output_path}")

def copy_metadata(source_image, target_image):
    """Copy all metadata from source to georeferenced image using ExifTool."""
    try:
        subprocess.run(["exiftool", "-overwrite_original", "-TagsFromFile", str(source_image), str(target_image)], check=True)
        print(f"✅ Metadata copied from {source_image} to {target_image}")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error copying metadata: {e}")


def process_images(input_folder, output_folder, hfov):
    """Process all images in a folder without stacking."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(".tif") and "stacked" not in filename.lower():
            input_path = Path(input_folder) / filename
            output_path = Path(output_folder) / f"georeferenced_{filename}"
            georeference_image(input_path, output_path, hfov)

# Define folders
input_folder = r"C:\\Tarun\\CIFRI_ponds\\0002_MX"
output_folder = r"C:\\Tarun\\CIFRI_ponds\\0002_MX\\geo"
hfov = 64  # Horizontal Field of View in degrees

# Run processing
process_images(input_folder, output_folder, hfov)


C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\osgeo\gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080051_0000_GRE.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080051_0000_NIR.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080051_0000_RED.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080051_0000_REG.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080059_0001_GRE.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080059_0001_NIR.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080059_0001_RED.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080059_0001_REG.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080110_0002_GRE.TIF
Georeferenced image saved: C:\Tarun\CIFRI_ponds\0002_MX

In [2]:
#rotation working.
import os
import numpy as np
import cv2
import subprocess
from pathlib import Path
from osgeo import gdal, osr
from exiftool import ExifTool

def extract_yaw(image_path):
    """Extract Yaw from georeferenced image metadata using ExifTool."""
    with ExifTool() as et:
        metadata = et.get_metadata(str(image_path))
    
    yaw = float(metadata.get("XMP:Yaw", 0))  # Default to 0 if missing
    return yaw

# def rotate_and_adjust(image, angle, original_size):
#     """Rotate image while keeping the final size exactly 1280x960 without shrinking or clipping."""
#     h, w = original_size  # Ensure output remains 1280x960
#     center = (w // 2, h // 2)
#     angle = -angle  # Convert to correct rotation direction

#     # Compute rotation matrix
#     rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)

#     # Compute bounding box after rotation
#     abs_cos = abs(rotation_matrix[0, 0])
#     abs_sin = abs(rotation_matrix[0, 1])
#     new_w = int(h * abs_sin + w * abs_cos)
#     new_h = int(h * abs_cos + w * abs_sin)

#     # Adjust transformation matrix to center
#     rotation_matrix[0, 2] += (new_w - w) / 2
#     rotation_matrix[1, 2] += (new_h - h) / 2

#     # Rotate image with expanded canvas
#     rotated = cv2.warpAffine(image, rotation_matrix, (new_w, new_h), 
#                              flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)

#     # Crop the rotated image to keep only the valid region
#     mask = (rotated > 0).astype(np.uint8)
#     coords = cv2.findNonZero(mask)
#     x, y, w_valid, h_valid = cv2.boundingRect(coords)
#     cropped = rotated[y:y + h_valid, x:x + w_valid]

#     # Resize the cropped region back to 1280x960 (if needed)
#     if cropped.shape[1] != w or cropped.shape[0] != h:
#         final_image = cv2.resize(cropped, (w, h), interpolation=cv2.INTER_CUBIC)
#     else:
#         final_image = cropped

#     return final_image

def ModifiedWay(image, angle):
    """
    Rotates an image without cropping any portion.
    """
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    angle = -angle

    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    abs_cos = abs(rotation_matrix[0, 0])
    abs_sin = abs(rotation_matrix[0, 1])

    new_w = int(h * abs_sin + w * abs_cos)
    new_h = int(h * abs_cos + w * abs_sin)

    rotation_matrix[0, 2] += (new_w / 2) - center[0]
    rotation_matrix[1, 2] += (new_h / 2) - center[1]

    rotated_image = cv2.warpAffine(image, rotation_matrix, (new_w, new_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    return rotated_image

# def apply_yaw_correction(image_path, output_path):
#     """Apply Yaw correction to a georeferenced image while ensuring size consistency."""
    
#     # Step 1: Extract Yaw angle
#     yaw = extract_yaw(image_path)
#     print(f"Yaw: {yaw}")

#     # Step 2: Open image using GDAL
#     ds = gdal.Open(str(image_path))
#     if ds is None:
#         print(f"❌ Error: Unable to open {image_path}")
#         return
    
#     # Convert GDAL image to OpenCV format
#     band = ds.GetRasterBand(1)
#     img_array = band.ReadAsArray()

#     # Step 3: Rotate image while keeping original size (1280x960)
#     rotated_img = ModifiedWay(img_array, yaw)#, (960, 1280))

#     # Step 4: Save the rotated image while preserving georeferencing
#     driver = gdal.GetDriverByName("GTiff")
#     out_ds = driver.Create(str(output_path), ds.RasterXSize, ds.RasterYSize, ds.RasterCount, band.DataType)

#     # Preserve geotransform and projection
#     out_ds.SetGeoTransform(ds.GetGeoTransform())
#     out_ds.SetProjection(ds.GetProjection())

#     # Write rotated data
#     out_ds.GetRasterBand(1).WriteArray(rotated_img)

#     # Close datasets
#     out_ds = None
#     ds = None
#     print(f"✅ Yaw-corrected image saved: {output_path}")

#     # Step 5: Copy metadata from original image to the rotated image
#     copy_metadata(image_path, output_path)

def apply_yaw_correction(image_path, output_path):
    """Apply Yaw correction to a georeferenced image while ensuring size consistency."""
    
    # Step 1: Extract Yaw angle
    yaw = extract_yaw(image_path)
    print(f"Yaw: {yaw}")

    # Step 2: Open image using GDAL
    ds = gdal.Open(str(image_path))
    if ds is None:
        print(f"❌ Error: Unable to open {image_path}")
        return
    
    # Convert GDAL image to OpenCV format
    band = ds.GetRasterBand(1)
    img_array = band.ReadAsArray()

    # Step 3: Rotate image while keeping original size (1280x960)
    rotated_img = ModifiedWay(img_array, yaw)

    # Step 4: Save the rotated image while preserving georeferencing
    driver = gdal.GetDriverByName("GTiff")
    
    # Use the dimensions of the rotated image for the output dataset
    rotated_height, rotated_width = rotated_img.shape
    out_ds = driver.Create(str(output_path), rotated_width, rotated_height, ds.RasterCount, band.DataType)

    # Preserve geotransform and projection
    out_ds.SetGeoTransform(ds.GetGeoTransform())
    out_ds.SetProjection(ds.GetProjection())

    # Write rotated data
    out_ds.GetRasterBand(1).WriteArray(rotated_img)

    # Close datasets
    out_ds = None
    ds = None
    print(f"✅ Yaw-corrected image saved: {output_path}")

    # Step 5: Copy metadata from original image to the rotated image
    copy_metadata(image_path, output_path)

def copy_metadata(source_image, target_image):
    """Copy all metadata from the original image to the yaw-corrected image using ExifTool."""
    try:
        subprocess.run([
            "exiftool",
            "-overwrite_original",
            "-TagsFromFile", str(source_image),
            str(target_image)
        ], check=True)
        print(f"✅ Metadata copied from {source_image} to {target_image}")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error copying metadata: {e}")

def process_images(input_folder, output_folder):
    """Process all georeferenced images in a folder for Yaw correction only."""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(".tif"):
            input_path = Path(input_folder) / filename
            output_path = Path(output_folder) / f"yaw_corrected_{filename}"
            apply_yaw_correction(input_path, output_path)

# Define folders
input_folder = r"C:\\Tarun\\CIFRI_ponds\\0002_MX\\geo"
output_folder = r"C:\\Tarun\\CIFRI_ponds\\0002_MX\\rotated\\yaw_corrected1"

# Run processing
process_images(input_folder, output_folder)


Yaw: -124.849533
✅ Yaw-corrected image saved: C:\Tarun\CIFRI_ponds\0002_MX\rotated\yaw_corrected1\yaw_corrected_georeferenced_IMG_241207_080051_0000_GRE.TIF
✅ Metadata copied from C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080051_0000_GRE.TIF to C:\Tarun\CIFRI_ponds\0002_MX\rotated\yaw_corrected1\yaw_corrected_georeferenced_IMG_241207_080051_0000_GRE.TIF
Yaw: -124.850143
✅ Yaw-corrected image saved: C:\Tarun\CIFRI_ponds\0002_MX\rotated\yaw_corrected1\yaw_corrected_georeferenced_IMG_241207_080051_0000_NIR.TIF
✅ Metadata copied from C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080051_0000_NIR.TIF to C:\Tarun\CIFRI_ponds\0002_MX\rotated\yaw_corrected1\yaw_corrected_georeferenced_IMG_241207_080051_0000_NIR.TIF
Yaw: -124.849831
✅ Yaw-corrected image saved: C:\Tarun\CIFRI_ponds\0002_MX\rotated\yaw_corrected1\yaw_corrected_georeferenced_IMG_241207_080051_0000_RED.TIF
✅ Metadata copied from C:\Tarun\CIFRI_ponds\0002_MX\geo\georeferenced_IMG_241207_080051_0000_RED.TI

In [10]:
#yaw pitch roll

import os
import rasterio
import numpy as np
from rasterio.transform import Affine
from scipy.ndimage import rotate as ndi_rotate
from exiftool import ExifTool  # Using ExifTool for metadata extraction

# Define input and output folders
input_folder = r"C:\\Tarun\\CIFRI_ponds\\0002_MX\\geo1"
output_folder = r"C:\\Tarun\\CIFRI_ponds\\0002_MX\\rotated\\orientation_corrected"

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

def extract_metadata(image_path):
    """Extract Yaw, Pitch, and Roll from image metadata using ExifTool."""
    with ExifTool() as et:
        metadata = et.get_metadata(image_path)

    yaw = float(metadata.get("XMP:Yaw", 0))
    pitch = float(metadata.get("XMP:Pitch", 0))
    roll = float(metadata.get("XMP:Roll", 0))

    return yaw, pitch, roll

# Process each image in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".tif") or filename.endswith(".TIF"):  # Process only TIFF files
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)

        print(f"Processing: {filename}")

        # Extract yaw, pitch, and roll from metadata
        yaw, pitch, roll = extract_metadata(input_path)
        print(f"Yaw: {yaw}, Pitch: {pitch}, Roll: {roll}")

        # Load the image and metadata
        with rasterio.open(input_path) as dataset:
            image = dataset.read(1)  # Read the first band
            profile = dataset.profile  # Preserve metadata
            original_transform = dataset.transform  # Get original transform
            crs = dataset.crs  # Get CRS

        # Apply Yaw, Pitch, and Roll corrections while preserving full image extent
        yaw_corrected_image = ndi_rotate(image, -yaw, reshape=True, mode='constant', cval=0)
        pitch_corrected_image = ndi_rotate(yaw_corrected_image, -pitch, reshape=True, mode='constant', cval=0)
        fully_corrected_image = ndi_rotate(pitch_corrected_image, -roll, reshape=True, mode='constant', cval=0)

        # Preserve the original georeferencing
        profile.update({
            "height": fully_corrected_image.shape[0],
            "width": fully_corrected_image.shape[1],
            "transform": original_transform,  # Preserve original geotransform
            "crs": crs  # Preserve original CRS
        })

        # Save the corrected image
        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(fully_corrected_image.astype(profile["dtype"]), 1)

        print(f"Saved: {output_path}")

print("Processing complete!")


Processing: georeferenced_IMG_241207_080128_0004_GRE.TIF
Yaw: -135.787689, Pitch: 6.398161, Roll: -10.084588
Saved: C:\\Tarun\\CIFRI_ponds\\0002_MX\\rotated\\orientation_corrected\georeferenced_IMG_241207_080128_0004_GRE.TIF
Processing: georeferenced_IMG_241207_080128_0004_NIR.TIF
Yaw: -135.787689, Pitch: 6.393958, Roll: -10.087056
Saved: C:\\Tarun\\CIFRI_ponds\\0002_MX\\rotated\\orientation_corrected\georeferenced_IMG_241207_080128_0004_NIR.TIF
Processing: georeferenced_IMG_241207_080128_0004_RED.TIF
Yaw: -135.787689, Pitch: 6.39603, Roll: -10.085839
Saved: C:\\Tarun\\CIFRI_ponds\\0002_MX\\rotated\\orientation_corrected\georeferenced_IMG_241207_080128_0004_RED.TIF
Processing: georeferenced_IMG_241207_080128_0004_REG.TIF
Yaw: -135.787689, Pitch: 6.402955, Roll: -10.081773
Saved: C:\\Tarun\\CIFRI_ponds\\0002_MX\\rotated\\orientation_corrected\georeferenced_IMG_241207_080128_0004_REG.TIF
Processing: georeferenced_IMG_241207_080130_0005_GRE.TIF
Yaw: -139.257034, Pitch: -7.88679, Roll: -6.

In [ ]:
#Bundle-adjustment 

import os
import numpy as np
import rasterio
import cv2
from rasterio.merge import merge
from rasterio.transform import Affine
from rasterio.warp import reproject, Resampling

def load_images_with_georef(image_folder):
    """
    Load 16-bit grayscale TIFF images and their georeferencing data.
    """
    image_data = []
    
    # List all files
    files = sorted(os.listdir(image_folder))
    if not files:
        raise ValueError(f"No files found in folder: {image_folder}")

    # Process each TIFF
    for file in files:
        file_path = os.path.join(image_folder, file)
        if not file.lower().endswith(".tif"):
            continue  # Skip non-TIFF files

        try:
            with rasterio.open(file_path) as src:
                if src.count < 1:  # Ensure at least one band exists
                    print(f"Skipping {file_path}: No bands detected")
                    continue
                
                image_array = src.read(1)  # Read the first band
                transform = src.transform
                bounds = src.bounds
                crs = src.crs
                metadata = src.profile
                
                image_data.append((image_array, transform, bounds, crs, metadata))
                print(f"Loaded: {file_path} | Shape: {image_array.shape} | CRS: {crs}")
        
        except rasterio.errors.RasterioIOError as e:
            print(f"Error opening {file_path}: {e}")

    if not image_data:
        raise ValueError(f"No valid TIFF images could be loaded from {image_folder}. Check files!")

    return image_data

def detect_features(image):
    sift = cv2.SIFT_create()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    keypoints, descriptors = sift.detectAndCompute(gray, None)
    return keypoints, descriptors

def detect_features_batch(images):
    """
    Detect keypoints and descriptors for a batch of images.
    """
    features = []
    sift = cv2.SIFT_create()

    for img_data in images:
        try:
            image_array = img_data[0]  # Extract the image array from the tuple
        except Exception as e:
            print(f"Error accessing image data: {e}")
            continue  # Skip this image

        # Convert 16-bit to 8-bit if needed
        if image_array.dtype == np.uint16:
            image_array = (image_array / 256).astype(np.uint8)

        keypoints, descriptors = sift.detectAndCompute(image_array, None)
        features.append((keypoints, descriptors))

    print(f"Detected features for {len(features)} images.")
    return features

def find_overlapping_images(images, index):
    """
    Find images that overlap with the reference image at 'index'.
    """
    ref_bounds = images[index][2]  # Extract 'bounds' from the tuple
    overlaps = []

    for i, img in enumerate(images):
        if i == index:
            continue  # Skip self-check

        img_bounds = img[2]  # Extract 'bounds' from the tuple
        if (
            ref_bounds.right > img_bounds.left
            and ref_bounds.left < img_bounds.right
            and ref_bounds.top > img_bounds.bottom
            and ref_bounds.bottom < img_bounds.top
        ):
            overlaps.append(i)

    return overlaps

def match_features_with_overlaps(images, features, index, overlaps, min_matches=4):
    matches = {}
    ref_kp, ref_des = features[index]
    flann = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=50))
    for i in overlaps:
        kp, des = features[i]
        if des is None:
            continue
        knn_matches = flann.knnMatch(ref_des, des, k=2)
        good_matches = [m for m, n in knn_matches if m.distance < 0.7 * n.distance]
        if len(good_matches) >= min_matches:
            matches[i] = good_matches
    return matches

def align_overlapping_images(images, matches, features, index, method="homography"):
    alignments = {index: np.eye(3)}  # Reference image is identity
    for i, good_matches in matches.items():
        kp_ref, _ = features[index]
        kp, _ = features[i]
        src_pts = np.float32([kp_ref[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        dst_pts = np.float32([kp[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        if method == "homography":
            matrix, _ = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
        else:
            matrix, _ = cv2.estimateAffinePartial2D(dst_pts, src_pts, method=cv2.RANSAC)
            matrix = np.vstack([matrix, [0, 0, 1]])
        if matrix is not None:
            alignments[i] = matrix
    return alignments

def apply_transform(image, transform):
    height, width = image.shape[1:]
    aligned_image = np.zeros_like(image, dtype=np.float32)
    for band in range(image.shape[0]):
        aligned_image[band] = cv2.warpPerspective(image[band], transform, (width, height), flags=cv2.INTER_LINEAR)
    return aligned_image

def create_mosaic(images, alignments, output_path):
    datasets = []
    ref_crs = images[0][3]  # Extract 'crs' from the tuple (4th element)
    
    for img in images:
        try:
            img_array, transform, bounds, crs, metadata = img  # Unpacking tuple
            datasets.append((img_array, transform, bounds, crs))
        except Exception as e:
            print(f"Skipping image due to error: {e}")

    # Further mosaic processing logic...

# Main Execution
image_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top"
output_path = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top\\mosaic\\final_large_mosaic_group.tif"
images = load_images_with_georef(image_folder)
print(f"Total images loaded: {len(images)}")
features = detect_features_batch(images)
all_alignments = {}
for idx in range(len(images)):
    overlaps = find_overlapping_images(images, idx)
    matches = match_features_with_overlaps(images, features, idx, overlaps)
    alignments = align_overlapping_images(images, matches, features, idx)
    all_alignments.update(alignments)
create_mosaic(images, all_alignments, output_path)

In [1]:
# mosaic working

import rasterio
import numpy as np
import traceback
import rasterio.merge
import rasterio.mask
from rasterio.transform import from_origin
from pathlib import Path
from skimage.exposure import rescale_intensity
from skimage.transform import resize
import matplotlib.pyplot as plt

def load_images_with_georef(folder):
    folder = Path(folder)  # Ensure it's a Path object
    images = []
    for file in sorted(folder.glob("*.tif")):
        with rasterio.open(file) as src:
            image_data = src.read(1)  # Read first band
            transform = src.transform
            bounds = src.bounds
            crs = src.crs
            metadata = src.meta
            images.append((image_data, transform, bounds, crs, metadata, file))
    return images

def detect_features_batch(images):
    print("Detecting features...")
    features = []
    for i, img_data in enumerate(images):
        print(f"Processing image {i}")
        features.append("dummy_features")  # Replace with actual feature detection
    print(f"Detected features for {len(features)} images.")
    return features

def find_overlapping_images(images, index):
    ref_bounds = images[index][2]  # Bounds (left, bottom, right, top)
    overlaps = []
    for i, img in enumerate(images):
        if i != index:
            img_bounds = img[2]
            if not (ref_bounds.right < img_bounds.left or ref_bounds.left > img_bounds.right or 
                    ref_bounds.top < img_bounds.bottom or ref_bounds.bottom > img_bounds.top):
                overlaps.append(i)
    print(f"Image {index} overlaps with: {overlaps}")
    return overlaps

def match_features_with_overlaps(images, features, index, overlaps):
    matches = {i: "dummy_match" for i in overlaps}  # Replace with real matching
    print(f"Feature matches for image {index}: {matches}")
    return matches

def align_overlapping_images(images, matches, features, index):
    alignments = {i: "dummy_alignment" for i in matches}  # Replace with actual alignment
    print(f"Alignments for image {index}: {alignments}")
    return alignments

def save_debug_image(image, filename):
    plt.imshow(image, cmap='gray')
    plt.axis('off')
    plt.savefig(filename, bbox_inches='tight')
    plt.close()

def blend_images(mosaic, masks):
    """Blend overlapping images using weighted feathering and resizing to match the largest image size."""
    max_height = max(img.shape[0] for img in mosaic)
    max_width = max(img.shape[1] for img in mosaic)
    
    resized_images = [resize(img, (max_height, max_width), preserve_range=True).astype(np.float32) for img in mosaic]
    resized_masks = [resize(mask, (max_height, max_width), preserve_range=True).astype(np.float32) for mask in masks]
    
    blended = np.zeros((max_height, max_width), dtype=np.float32)
    weight_sum = np.zeros((max_height, max_width), dtype=np.float32)
    
    for idx, (img, mask) in enumerate(zip(resized_images, resized_masks)):
        weight = rescale_intensity(mask, out_range=(0.1, 1.0))
        save_debug_image(img, f"debug_img_{idx}.png")
        save_debug_image(mask, f"debug_mask_{idx}.png")
        blended += img * weight
        weight_sum += weight
    
    weight_sum[weight_sum == 0] = 1  # Prevent division by zero
    return (blended / weight_sum).astype(mosaic[0].dtype)

def create_mosaic(images, alignments, output_path):
    try:
        if not images:
            print("⚠ No images available for mosaicking.")
            return
        
        datasets = [rasterio.open(img[5]) for img in images]
        mosaic, out_transform = rasterio.merge.merge(datasets)
        
        masks = [ds.read_masks(1) for ds in datasets]
        mosaic = blend_images(mosaic, masks)
        
        out_meta = datasets[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[0],
            "width": mosaic.shape[1],
            "transform": out_transform
        })
        
        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(mosaic, 1)
        
        print("✅ Seamless mosaic generation completed and saved at:", output_path)
    except Exception as e:
        print("⚠ Error during mosaic creation:")
        traceback.print_exc()

# Main Execution
image_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top"
output_path = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top\\mosaic\\final_large_mosaic_group2.tif"

images = load_images_with_georef(image_folder)
print(f"Total images loaded: {len(images)}")

if len(images) == 0:
    print("⚠ No images found in the folder!")
else:
    features = detect_features_batch(images)
    all_alignments = {}
    
    for idx in range(len(images)):
        overlaps = find_overlapping_images(images, idx)
        matches = match_features_with_overlaps(images, features, idx, overlaps)
        alignments = align_overlapping_images(images, matches, features, idx)
        all_alignments.update(alignments)
    
    print(f"Total alignments: {len(all_alignments)}")
    create_mosaic(images, all_alignments, output_path)

# # Main Execution
# image_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top"
# output_path = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top\\mosaic\\final_large_mosaic_group2.tif"

# images = load_images_with_georef(image_folder)
# print(f"Total images loaded: {len(images)}")

# if len(images) == 0:
#     print("⚠ No images found in the folder!")
# else:
#     features = detect_features_batch(images)
#     all_alignments = {}
    
#     for idx in range(len(images)):
#         overlaps = find_overlapping_images(images, idx)
#         matches = match_features_with_overlaps(images, features, idx, overlaps)
#         alignments = align_overlapping_images(images, matches, features, idx)
#         all_alignments.update(alignments)
    
#     print(f"Total alignments: {len(all_alignments)}")
#     create_mosaic(images, all_alignments, output_path)

Total images loaded: 45
Detecting features...
Processing image 0
Processing image 1
Processing image 2
Processing image 3
Processing image 4
Processing image 5
Processing image 6
Processing image 7
Processing image 8
Processing image 9
Processing image 10
Processing image 11
Processing image 12
Processing image 13
Processing image 14
Processing image 15
Processing image 16
Processing image 17
Processing image 18
Processing image 19
Processing image 20
Processing image 21
Processing image 22
Processing image 23
Processing image 24
Processing image 25
Processing image 26
Processing image 27
Processing image 28
Processing image 29
Processing image 30
Processing image 31
Processing image 32
Processing image 33
Processing image 34
Processing image 35
Processing image 36
Processing image 37
Processing image 38
Processing image 39
Processing image 40
Processing image 41
Processing image 42
Processing image 43
Processing image 44
Detected features for 45 images.
Image 0 overlaps with: [1, 2, 3

In [2]:
# Mosaic and Blend working

import rasterio
import numpy as np
import traceback
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from pathlib import Path
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt

def load_images_with_georef(folder):
    folder = Path(folder)
    images = []
    for file in sorted(folder.glob("*.tif")):
        with rasterio.open(file) as src:
            image_data = src.read(1)
            transform = src.transform
            bounds = src.bounds
            crs = src.crs
            metadata = src.meta.copy()
            images.append((image_data, transform, bounds, crs, metadata, file))
    return images

def create_mosaic(images, output_path):
    try:
        if not images:
            print("⚠ No images available for mosaicking.")
            return
        
        # Calculate union bounds of all images
        all_bounds = [img[2] for img in images]
        left = min(b.left for b in all_bounds)
        right = max(b.right for b in all_bounds)
        bottom = min(b.bottom for b in all_bounds)
        top = max(b.top for b in all_bounds)
        
        # Get resolution from first image (assumes all images have same resolution)
        first_transform = images[0][1]
        res_x = first_transform.a
        res_y = abs(first_transform.e)
        
        # Calculate output dimensions
        width = int((right - left) / res_x)
        height = int((top - bottom) / res_y)
        
        # Create output transform
        output_transform = from_origin(left, top, res_x, res_y)
        
        # Initialize mosaic arrays
        mosaic_data = np.zeros((height, width), dtype=np.float32)
        mosaic_weights = np.zeros((height, width), dtype=np.float32)
        
        for img_tuple in images:
            img_data, src_transform, _, src_crs, metadata, _ = img_tuple
            
            # Reproject image into mosaic grid
            dst_data = np.zeros((height, width), dtype=np.float32)
            reproject(
                source=img_data,
                destination=dst_data,
                src_transform=src_transform,
                src_crs=src_crs,
                dst_transform=output_transform,
                dst_crs=src_crs,
                resampling=Resampling.bilinear,
                src_nodata=0,
                dst_nodata=0
            )
            
            # Create validity mask and calculate feather weights
            valid_mask = (dst_data != 0).astype(np.float32)
            if valid_mask.sum() == 0:
                continue
            
            # Calculate distance transform for feathering
            distance = distance_transform_edt(valid_mask)
            max_distance = np.max(distance)
            if max_distance == 0:
                weights = valid_mask
            else:
                weights = (distance / max_distance) * valid_mask
            
            # Accumulate data and weights
            mosaic_data += dst_data * weights
            mosaic_weights += weights
        
        # Normalize mosaic data
        mosaic_weights[mosaic_weights == 0] = 1e-9  # Avoid division by zero
        mosaic = (mosaic_data / mosaic_weights).astype(images[0][0].dtype)
        
        # Prepare output metadata
        out_meta = images[0][4]
        out_meta.update({
            "driver": "GTiff",
            "height": height,
            "width": width,
            "transform": output_transform,
            "crs": src_crs
        })
        
        # Write output
        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(mosaic, 1)
        
        print(f"✅ Mosaic successfully created at {output_path}")
    
    except Exception as e:
        print("⚠ Error during mosaic creation:")
        traceback.print_exc()

# Main Execution
image_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top"
output_path = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top\\mosaic\\final_large_mosaic_group2.tif"

images = load_images_with_georef(image_folder)
if not images:
    print("No images found!")
else:
    create_mosaic(images, output_path)

✅ Mosaic successfully created at C:\Tarun\CIFRI_ponds\0002_MX\secondside\green\top\mosaic\final_large_mosaic_group2.tif


In [4]:
#seamless mosaic working

import rasterio
import numpy as np
import traceback
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from pathlib import Path
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt

def load_images_with_georef(folder):
    folder = Path(folder)
    return [rasterio.open(file) for file in sorted(folder.glob("*.tif"))]

def calculate_weights(dataset, output_transform, output_shape):
    """Calculate distance-based weights with edge detection"""
    # Create validity mask
    mask = np.zeros(output_shape, dtype=np.uint8)
    reproject(
        source=rasterio.band(dataset, 1),
        destination=mask,
        src_transform=dataset.transform,
        src_crs=dataset.crs,
        dst_transform=output_transform,
        dst_crs=dataset.crs,
        resampling=Resampling.nearest
    )
    
    # Calculate distance transform
    distances = distance_transform_edt(mask)
    max_distance = np.max(distances)
    return distances / max_distance if max_distance > 0 else mask

def smart_blend(mosaic_data, mosaic_weights, output_shape, method='sharpest'):
    """Advanced blending techniques"""
    if method == 'feather':
        return mosaic_data / mosaic_weights
    
    # Initialize output
    blended = np.zeros(output_shape, dtype=mosaic_data.dtype)
    
    # Find maximum weight indices
    max_weight_indices = np.argmax(mosaic_weights, axis=-1)
    
    if method == 'max_weight':
        for i in range(mosaic_data.shape[-1]):
            mask = max_weight_indices == i
            blended[mask] = mosaic_data[..., i][mask]
    elif method == 'sharpest':
        # Placeholder for sharpness detection - could use gradient magnitude
        sharpness = np.sum(np.abs(np.gradient(mosaic_data, axis=(0, 1))), axis=-1)
        sharpest = np.argmax(sharpness, axis=-1)
        blended = np.take_along_axis(mosaic_data, sharpest[..., None], axis=-1)[..., 0]
    
    return blended

def create_advanced_mosaic(datasets, output_path, blend_method='feather'):
    try:
        # Calculate output bounds and resolution
        all_bounds = [ds.bounds for ds in datasets]
        res = datasets[0].res
        output_transform = from_origin(
            min(b.left for b in all_bounds),
            max(b.top for b in all_bounds),
            res[0],
            res[1]
        )
        output_shape = (
            int((max(b.top for b in all_bounds) - min(b.bottom for b in all_bounds)) // res[1]),
            int((max(b.right for b in all_bounds) - min(b.left for b in all_bounds)) // res[0])
        )

        # Initialize storage for blending
        num_images = len(datasets)
        mosaic_stack = np.zeros(output_shape + (num_images,), dtype=np.float32)
        weight_stack = np.zeros(output_shape + (num_images,), dtype=np.float32)

        # Process each image
        for i, ds in enumerate(datasets):
            print(f"Processing image {i+1}/{num_images}")
            
            # Reproject data
            data = np.zeros(output_shape, dtype=np.float32)
            reproject(
                source=rasterio.band(ds, 1),
                destination=data,
                src_transform=ds.transform,
                src_crs=ds.crs,
                dst_transform=output_transform,
                dst_crs=ds.crs,
                resampling=Resampling.cubic
            )
            
            # Calculate weights
            weights = calculate_weights(ds, output_transform, output_shape)
            
            # Store in stacks
            mosaic_stack[..., i] = data
            weight_stack[..., i] = weights

            # Debug output
            plt.imsave(f'debug_{i}_data.png', data, cmap='gray')
            plt.imsave(f'debug_{i}_weights.png', weights, cmap='gray')

        # Apply blending strategy
        if blend_method == 'feather':
            total_weights = np.sum(weight_stack, axis=-1)
            total_weights[total_weights == 0] = 1e-9
            mosaic = np.sum(mosaic_stack * weight_stack, axis=-1) / total_weights
        else:
            mosaic = smart_blend(mosaic_stack, weight_stack, output_shape, blend_method)

        # Convert data type for saving
        meta = datasets[0].meta.copy()
        meta.update({
            'driver': 'GTiff',
            'height': output_shape[0],
            'width': output_shape[1],
            'transform': output_transform,
            'dtype': 'float32'  # Ensure correct data type
        })

        with rasterio.open(output_path, 'w', **meta) as dst:
            dst.write(mosaic.astype('float32'), 1)

        print(f"✅ Advanced mosaic created using {blend_method} blending: {output_path}")

    except Exception as e:
        print(f"⚠ Error: {str(e)}")
        traceback.print_exc()
import rasterio
import numpy as np
import traceback
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from pathlib import Path
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt

def load_images_with_georef(folder):
    folder = Path(folder)
    return [rasterio.open(file) for file in sorted(folder.glob("*.tif"))]

def calculate_weights(dataset, output_transform, output_shape):
    """Calculate distance-based weights with edge detection"""
    # Create validity mask
    mask = np.zeros(output_shape, dtype=np.uint8)
    reproject(
        source=rasterio.band(dataset, 1),
        destination=mask,
        src_transform=dataset.transform,
        src_crs=dataset.crs,
        dst_transform=output_transform,
        dst_crs=dataset.crs,
        resampling=Resampling.nearest
    )
    
    # Calculate distance transform
    distances = distance_transform_edt(mask)
    max_distance = np.max(distances)
    return distances / max_distance if max_distance > 0 else mask

def smart_blend(mosaic_data, mosaic_weights, output_shape, method='sharpest'):
    """Advanced blending techniques"""
    if method == 'feather':
        return mosaic_data / mosaic_weights
    
    # Initialize output
    blended = np.zeros(output_shape, dtype=mosaic_data.dtype)
    
    # Find maximum weight indices
    max_weight_indices = np.argmax(mosaic_weights, axis=-1)
    
    if method == 'max_weight':
        for i in range(mosaic_data.shape[-1]):
            mask = max_weight_indices == i
            blended[mask] = mosaic_data[..., i][mask]
    elif method == 'sharpest':
        # Placeholder for sharpness detection - could use gradient magnitude
        sharpness = np.sum(np.abs(np.gradient(mosaic_data, axis=(0, 1))), axis=-1)
        sharpest = np.argmax(sharpness, axis=-1)
        blended = np.take_along_axis(mosaic_data, sharpest[..., None], axis=-1)[..., 0]
    
    return blended

def create_advanced_mosaic(datasets, output_path, blend_method='feather'):
    try:
        # Calculate output bounds and resolution
        all_bounds = [ds.bounds for ds in datasets]
        res = datasets[0].res
        output_transform = from_origin(
            min(b.left for b in all_bounds),
            max(b.top for b in all_bounds),
            res[0],
            res[1]
        )
        output_shape = (
            int((max(b.top for b in all_bounds) - min(b.bottom for b in all_bounds)) // res[1]),
            int((max(b.right for b in all_bounds) - min(b.left for b in all_bounds)) // res[0])
        )

        # Initialize storage for blending
        num_images = len(datasets)
        mosaic_stack = np.zeros(output_shape + (num_images,), dtype=np.float32)
        weight_stack = np.zeros(output_shape + (num_images,), dtype=np.float32)

        # Process each image
        for i, ds in enumerate(datasets):
            print(f"Processing image {i+1}/{num_images}")
            
            # Reproject data
            data = np.zeros(output_shape, dtype=np.float32)
            reproject(
                source=rasterio.band(ds, 1),
                destination=data,
                src_transform=ds.transform,
                src_crs=ds.crs,
                dst_transform=output_transform,
                dst_crs=ds.crs,
                resampling=Resampling.cubic
            )
            
            # Calculate weights
            weights = calculate_weights(ds, output_transform, output_shape)
            
            # Store in stacks
            mosaic_stack[..., i] = data
            weight_stack[..., i] = weights

            # Debug output
            plt.imsave(f'debug_{i}_data.png', data, cmap='gray')
            plt.imsave(f'debug_{i}_weights.png', weights, cmap='gray')

        # Apply blending strategy
        if blend_method == 'feather':
            total_weights = np.sum(weight_stack, axis=-1)
            total_weights[total_weights == 0] = 1e-9
            mosaic = np.sum(mosaic_stack * weight_stack, axis=-1) / total_weights
        else:
            mosaic = smart_blend(mosaic_stack, weight_stack, output_shape, blend_method)

        # Convert data type for saving
        meta = datasets[0].meta.copy()
        meta.update({
            'driver': 'GTiff',
            'height': output_shape[0],
            'width': output_shape[1],
            'transform': output_transform,
            'dtype': 'float32'  # Ensure correct data type
        })

        with rasterio.open(output_path, 'w', **meta) as dst:
            dst.write(mosaic.astype('float32'), 1)

        print(f"✅ Advanced mosaic created using {blend_method} blending: {output_path}")

    except Exception as e:
        print(f"⚠ Error: {str(e)}")
        traceback.print_exc()

# Usage
# image_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top"
# output_path = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top\\mosaic\\final_mosaic.tif"

image_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\test"
output_path = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\mosaic\\testmosaic.tif"

datasets = load_images_with_georef(image_folder)
if datasets:
    create_advanced_mosaic(datasets, output_path, blend_method='max_weight')
else:
    print("No datasets found!")

Processing image 1/10
Processing image 2/10
Processing image 3/10
Processing image 4/10
Processing image 5/10
Processing image 6/10
Processing image 7/10
Processing image 8/10
Processing image 9/10
Processing image 10/10
✅ Advanced mosaic created using max_weight blending: C:\Tarun\CIFRI_ponds\0002_MX\secondside\Red\mosaic\testmosaic.tif


In [7]:
import rasterio
import numpy as np
import traceback
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from pathlib import Path
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt

def load_images_with_georef(folder):
    folder = Path(folder)
    return [rasterio.open(file) for file in sorted(folder.glob("*.tif"))]

def calculate_weights(dataset, output_transform, output_shape):
    """Calculate distance-based weights with edge detection"""
    # Create validity mask
    mask = np.zeros(output_shape, dtype=np.uint8)
    reproject(
        source=rasterio.band(dataset, 1),
        destination=mask,
        src_transform=dataset.transform,
        src_crs=dataset.crs,
        dst_transform=output_transform,
        dst_crs=dataset.crs,
        resampling=Resampling.nearest
    )
    
    # Calculate distance transform
    distances = distance_transform_edt(mask)
    max_distance = np.max(distances)
    return distances / max_distance if max_distance > 0 else mask

def smart_blend(mosaic_data, mosaic_weights, output_shape, method='sharpest'):
    """Advanced blending techniques"""
    if method == 'feather':
        return mosaic_data / mosaic_weights
    
    # Initialize output
    blended = np.zeros(output_shape, dtype=mosaic_data.dtype)
    
    # Find maximum weight indices
    max_weight_indices = np.argmax(mosaic_weights, axis=-1)
    
    if method == 'max_weight':
        for i in range(mosaic_data.shape[-1]):
            mask = max_weight_indices == i
            blended[mask] = mosaic_data[..., i][mask]
    elif method == 'sharpest':
        # Placeholder for sharpness detection - could use gradient magnitude
        sharpness = np.sum(np.abs(np.gradient(mosaic_data, axis=(0, 1))), axis=-1)
        sharpest = np.argmax(sharpness, axis=-1)
        blended = np.take_along_axis(mosaic_data, sharpest[..., None], axis=-1)[..., 0]
    
    return blended

def create_advanced_mosaic(datasets, output_path, blend_method='feather'):
    try:
        # Calculate output bounds and resolution
        all_bounds = [ds.bounds for ds in datasets]
        res = datasets[0].res
        output_transform = from_origin(
            min(b.left for b in all_bounds),
            max(b.top for b in all_bounds),
            res[0],
            res[1]
        )
        output_shape = (
            int((max(b.top for b in all_bounds) - min(b.bottom for b in all_bounds)) // res[1]),
            int((max(b.right for b in all_bounds) - min(b.left for b in all_bounds)) // res[0])
        )

        # Initialize storage for blending
        num_images = len(datasets)
        mosaic_stack = np.zeros(output_shape + (num_images,), dtype=np.float32)
        weight_stack = np.zeros(output_shape + (num_images,), dtype=np.float32)

        # Process each image
        for i, ds in enumerate(datasets):
            print(f"Processing image {i+1}/{num_images}")
            
            # Reproject data
            data = np.zeros(output_shape, dtype=np.float32)
            reproject(
                source=rasterio.band(ds, 1),
                destination=data,
                src_transform=ds.transform,
                src_crs=ds.crs,
                dst_transform=output_transform,
                dst_crs=ds.crs,
                resampling=Resampling.cubic
            )
            
            # Calculate weights
            weights = calculate_weights(ds, output_transform, output_shape)
            
            # Store in stacks
            mosaic_stack[..., i] = data
            weight_stack[..., i] = weights

            # Debug output
            plt.imsave(f'debug_{i}_data.png', data, cmap='gray')
            plt.imsave(f'debug_{i}_weights.png', weights, cmap='gray')

        # Apply blending strategy
        if blend_method == 'feather':
            total_weights = np.sum(weight_stack, axis=-1)
            total_weights[total_weights == 0] = 1e-9
            mosaic = np.sum(mosaic_stack * weight_stack, axis=-1) / total_weights
        else:
            mosaic = smart_blend(mosaic_stack, weight_stack, output_shape, blend_method)

        # Convert data type for saving
        meta = datasets[0].meta.copy()
        meta.update({
            'driver': 'GTiff',
            'height': output_shape[0],
            'width': output_shape[1],
            'transform': output_transform,
            'dtype': 'float32'  # Ensure correct data type
        })

        with rasterio.open(output_path, 'w', **meta) as dst:
            dst.write(mosaic.astype('float32'), 1)

        print(f"✅ Advanced mosaic created using {blend_method} blending: {output_path}")

    except Exception as e:
        print(f"⚠ Error: {str(e)}")
        traceback.print_exc()
import rasterio
import numpy as np
import traceback
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from pathlib import Path
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt

def load_images_with_georef(folder):
    folder = Path(folder)
    return [rasterio.open(file) for file in sorted(folder.glob("*.tif"))]

def calculate_edge_weights(dataset, output_transform, output_shape):
    """Calculate edge-based feathering weights for blending"""
    # Create a mask where valid pixels exist
    mask = np.zeros(output_shape, dtype=np.uint8)
    reproject(
        source=rasterio.band(dataset, 1),
        destination=mask,
        src_transform=dataset.transform,
        src_crs=dataset.crs,
        dst_transform=output_transform,
        dst_crs=dataset.crs,
        resampling=Resampling.nearest
    )
    
    # Invert mask: 1 for background, 0 for valid pixels
    edge_mask = 1 - mask  
    
    # Compute distance transform (distance to the nearest edge)
    distances = distance_transform_edt(edge_mask)
    
    # Normalize weights to the range [0, 1]
    max_distance = np.max(distances)
    weights = distances / max_distance if max_distance > 0 else mask
    
    # Invert weights so edges get high values, centers remain unchanged
    return 1 - weights  

def smart_blend(mosaic_data, mosaic_weights, output_shape, method='sharpest'):
    """Advanced blending techniques"""
    if method == 'feather':
        return mosaic_data / mosaic_weights
    
    # Initialize output
    blended = np.zeros(output_shape, dtype=mosaic_data.dtype)
    
    # Find maximum weight indices
    max_weight_indices = np.argmax(mosaic_weights, axis=-1)
    
    if method == 'max_weight':
        for i in range(mosaic_data.shape[-1]):
            mask = max_weight_indices == i
            blended[mask] = mosaic_data[..., i][mask]
    elif method == 'sharpest':
        # Placeholder for sharpness detection - could use gradient magnitude
        sharpness = np.sum(np.abs(np.gradient(mosaic_data, axis=(0, 1))), axis=-1)
        sharpest = np.argmax(sharpness, axis=-1)
        blended = np.take_along_axis(mosaic_data, sharpest[..., None], axis=-1)[..., 0]
    
    return blended

def create_advanced_mosaic(datasets, output_path, blend_method='feather'):
    try:
        # Calculate output bounds and resolution
        all_bounds = [ds.bounds for ds in datasets]
        res = datasets[0].res
        output_transform = from_origin(
            min(b.left for b in all_bounds),
            max(b.top for b in all_bounds),
            res[0],
            res[1]
        )
        output_shape = (
            int((max(b.top for b in all_bounds) - min(b.bottom for b in all_bounds)) // res[1]),
            int((max(b.right for b in all_bounds) - min(b.left for b in all_bounds)) // res[0])
        )

        # Initialize storage for blending
        num_images = len(datasets)
        mosaic_stack = np.zeros(output_shape + (num_images,), dtype=np.float32)
        weight_stack = np.zeros(output_shape + (num_images,), dtype=np.float32)

        # Process each image
        for i, ds in enumerate(datasets):
            print(f"Processing image {i+1}/{num_images}")
            
            # Reproject data
            data = np.zeros(output_shape, dtype=np.float32)
            reproject(
                source=rasterio.band(ds, 1),
                destination=data,
                src_transform=ds.transform,
                src_crs=ds.crs,
                dst_transform=output_transform,
                dst_crs=ds.crs,
                resampling=Resampling.cubic
            )
            
            # Calculate weights
            # weights = calculate_weights(ds, output_transform, output_shape)
            weights = calculate_edge_weights(ds, output_transform, output_shape)

            
            # Store in stacks
            mosaic_stack[..., i] = data
            weight_stack[..., i] = weights

            # Debug output
            plt.imsave(f'debug_{i}_data.png', data, cmap='gray')
            plt.imsave(f'debug_{i}_weights.png', weights, cmap='gray')

        # Apply blending strategy
        if blend_method == 'feather':
            total_weights = np.sum(weight_stack, axis=-1)
            total_weights[total_weights == 0] = 1e-9
            mosaic = np.sum(mosaic_stack * weight_stack, axis=-1) / total_weights
        else:
            mosaic = smart_blend(mosaic_stack, weight_stack, output_shape, blend_method)

        # Convert data type for saving
        meta = datasets[0].meta.copy()
        meta.update({
            'driver': 'GTiff',
            'height': output_shape[0],
            'width': output_shape[1],
            'transform': output_transform,
            'dtype': 'float32'  # Ensure correct data type
        })

        with rasterio.open(output_path, 'w', **meta) as dst:
            dst.write(mosaic.astype('float32'), 1)

        print(f"✅ Advanced mosaic created using {blend_method} blending: {output_path}")

    except Exception as e:
        print(f"⚠ Error: {str(e)}")
        traceback.print_exc()

# Usage
image_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top"
output_path = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top\\mosaic\\final_mosaic.tif"

datasets = load_images_with_georef(image_folder)
if datasets:
    create_advanced_mosaic(datasets, output_path, blend_method='max_weight')
else:
    print("No datasets found!")

# Usage
image_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top"
output_path = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\green\\top\\mosaic\\final_mosaic1.tif"

datasets = load_images_with_georef(image_folder)
if datasets:
    create_advanced_mosaic(datasets, output_path, blend_method='max_weight')
else:
    print("No datasets found!")


Processing image 1/45
Processing image 2/45
Processing image 3/45
Processing image 4/45
Processing image 5/45
Processing image 6/45
Processing image 7/45
Processing image 8/45
Processing image 9/45
Processing image 10/45
Processing image 11/45
Processing image 12/45
Processing image 13/45
Processing image 14/45
Processing image 15/45
Processing image 16/45
Processing image 17/45
Processing image 18/45
Processing image 19/45
Processing image 20/45
Processing image 21/45
Processing image 22/45
Processing image 23/45
Processing image 24/45
Processing image 25/45
Processing image 26/45
Processing image 27/45
Processing image 28/45
Processing image 29/45
Processing image 30/45
Processing image 31/45
Processing image 32/45
Processing image 33/45
Processing image 34/45
Processing image 35/45
Processing image 36/45
Processing image 37/45
Processing image 38/45
Processing image 39/45
Processing image 40/45
Processing image 41/45
Processing image 42/45
Processing image 43/45
Processing image 44/

Traceback (most recent call last):
  File "C:\Users\BDL\AppData\Local\Temp\ipykernel_3448\982194907.py", line 261, in create_advanced_mosaic
    with rasterio.open(output_path, 'w', **meta) as dst:
  File "C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\env.py", line 451, in wrapper
    return f(*args, **kwds)
  File "C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\__init__.py", line 343, in open
    dataset = writer(
  File "rasterio\\_io.pyx", line 1466, in rasterio._io.DatasetWriterBase.__init__
  File "rasterio\\_io.pyx", line 332, in rasterio._io._delete_dataset_if_exists
  File "rasterio\\_err.pyx", line 195, in rasterio._err.exc_wrap_int
rasterio._err.CPLE_AppDefinedError: Deleting C:/Tarun/CIFRI_ponds/0002_MX/secondside/green/top/mosaic/final_mosaic.tif failed: Permission denied


Processing image 1/45
Processing image 2/45
Processing image 3/45
Processing image 4/45
Processing image 5/45
Processing image 6/45
Processing image 7/45
Processing image 8/45
Processing image 9/45
Processing image 10/45
Processing image 11/45
Processing image 12/45
Processing image 13/45
Processing image 14/45
Processing image 15/45
Processing image 16/45
Processing image 17/45
Processing image 18/45
Processing image 19/45
Processing image 20/45
Processing image 21/45
Processing image 22/45
Processing image 23/45
Processing image 24/45
Processing image 25/45
Processing image 26/45
Processing image 27/45
Processing image 28/45
Processing image 29/45
Processing image 30/45
Processing image 31/45
Processing image 32/45
Processing image 33/45
Processing image 34/45
Processing image 35/45
Processing image 36/45
Processing image 37/45
Processing image 38/45
Processing image 39/45
Processing image 40/45
Processing image 41/45
Processing image 42/45
Processing image 43/45
Processing image 44/

In [30]:
#CGT
import numpy as np
import cv2
import rasterio
import glob
import os
from rasterio.transform import Affine
from rasterio.merge import merge

def align_images(img1, img2):
    img1_scaled = cv2.normalize(img1, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    img2_scaled = cv2.normalize(img2, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    orb = cv2.ORB_create(5000)
    kp1, des1 = orb.detectAndCompute(img1_scaled, None)
    kp2, des2 = orb.detectAndCompute(img2_scaled, None)
    
    if des1 is None or des2 is None or len(des1) < 10 or len(des2) < 10:
        raise ValueError("Feature detection failed or insufficient features found.")
    
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)
    
    if len(matches) < 10:
        raise ValueError("Not enough good matches to compute transformation.")
    
    src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    
    M, mask = cv2.estimateAffinePartial2D(dst_pts, src_pts, method=cv2.RANSAC, ransacReprojThreshold=3.0)
    return M

def mosaic_images(image_paths, output_path):
    if len(image_paths) < 2:
        raise ValueError("Not enough images to create a mosaic.")
    
    raster_datasets = [rasterio.open(path) for path in image_paths]
    
    # Check CRS consistency
    crs_list = [ds.crs for ds in raster_datasets]
    if len(set(crs_list)) > 1:
        raise ValueError("CRS mismatch between images.")
    
    base_img = raster_datasets[0].read(1)
    base_transform = raster_datasets[0].transform
    base_profile = raster_datasets[0].profile
    
    transformed_images = [base_img]
    transformed_transforms = [base_transform]
    
    for i in range(1, len(raster_datasets)):
        img = raster_datasets[i].read(1)
        M = align_images(base_img, img)
        
        if M is not None:
            h, w = img.shape
            aligned_img = cv2.warpAffine(img, M, (w, h))
            transformed_images.append(aligned_img)
        else:
            transformed_images.append(img)
    
    mosaic, out_transform = merge(raster_datasets, method='first')
    
    print(f"Mosaic shape before writing: {mosaic.shape}")
    print(f"Output transform: {out_transform}")
    
    if mosaic.ndim == 2:
        mosaic = np.expand_dims(mosaic, axis=0)  # Ensure a band dimension
    
    if mosaic.ndim != 3:
        raise ValueError(f"Unexpected mosaic shape: {mosaic.shape}")
    
    base_profile.update(
        dtype=rasterio.uint16,
        count=mosaic.shape[0],
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=out_transform
    )
    
    with rasterio.open(output_path, 'w', **base_profile) as dst:
        for i in range(mosaic.shape[0]):
            dst.write(mosaic[i].astype(rasterio.uint16), indexes=i+1)
    
    print(f"Mosaic saved at {output_path}")

# Example usage
input_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\test"  # Update with actual path
output_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\mosaic\\"  # Update with actual output path
image_paths = sorted(glob.glob(os.path.join(input_folder, "*.TIF")))
if len(image_paths) > 1:
    mosaic_images(image_paths, output_path)
else:
    print("Not enough images for mosaicking.")

Mosaic shape before writing: (1, 2642, 1758)
Output transform: | 0.00, 0.00, 88.47|
| 0.00,-0.00, 22.95|
| 0.00, 0.00, 1.00|
Mosaic saved at C:\Tarun\CIFRI_ponds\0002_MX\secondside\Red\mosaic\testmosaic.tif


In [31]:
#seek
import numpy as np
import cv2
import rasterio
from rasterio.transform import Affine
from rasterio.merge import merge
from rasterio.io import MemoryFile
import glob
import os

def align_images(img1, img2):
    img1_scaled = cv2.normalize(img1, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    img2_scaled = cv2.normalize(img2, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    orb = cv2.ORB_create(5000)
    kp1, des1 = orb.detectAndCompute(img1_scaled, None)
    kp2, des2 = orb.detectAndCompute(img2_scaled, None)
    
    if des1 is None or des2 is None or len(des1) < 10 or len(des2) < 10:
        raise ValueError("Feature detection failed or insufficient features found.")
    
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)
    
    if len(matches) < 10:
        raise ValueError("Not enough good matches to compute transformation.")
    
    src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    
    M, mask = cv2.estimateAffinePartial2D(dst_pts, src_pts, method=cv2.RANSAC, ransacReprojThreshold=3.0)
    return M

def mosaic_images(image_paths, output_path):
    if len(image_paths) < 2:
        raise ValueError("Not enough images to create a mosaic.")
    
    raster_datasets = [rasterio.open(path) for path in image_paths]
    
    # Check CRS consistency
    crs_list = [ds.crs for ds in raster_datasets]
    if len(set(crs_list)) > 1:
        raise ValueError("CRS mismatch between images.")
    
    base_img = raster_datasets[0].read(1)
    base_profile = raster_datasets[0].profile
    
    datasets_to_merge = [raster_datasets[0]]  # Start with the base image
    
    for i in range(1, len(raster_datasets)):
        img = raster_datasets[i].read(1)
        original_transform = raster_datasets[i].transform
        
        try:
            M = align_images(base_img, img)
        except Exception as e:
            print(f"Skipping image {i} due to alignment error: {e}")
            continue
        
        # Warp the image using M
        h, w = img.shape
        aligned_img = cv2.warpAffine(img, M, (w, h))
        
        # Compute new affine transform parameters
        a_orig = original_transform.a
        b_orig = original_transform.b
        c_orig = original_transform.c
        d_orig = original_transform.d
        e_orig = original_transform.e
        f_orig = original_transform.f
        
        M00, M01, M02 = M[0, 0], M[0, 1], M[0, 2]
        M10, M11, M12 = M[1, 0], M[1, 1], M[1, 2]
        
        a_new = a_orig * M00 + b_orig * M10
        b_new = a_orig * M01 + b_orig * M11
        c_new = a_orig * M02 + b_orig * M12 + c_orig
        d_new = d_orig * M00 + e_orig * M10
        e_new = d_orig * M01 + e_orig * M11
        f_new = d_orig * M02 + e_orig * M12 + f_orig
        
        new_transform = Affine(a_new, b_new, c_new, d_new, e_new, f_new)
        
        # Create an in-memory dataset for the aligned image
        profile = raster_datasets[i].profile
        profile.update({
            'transform': new_transform,
            'height': h,
            'width': w,
            'dtype': 'uint16'
        })
        
        with MemoryFile() as memfile:
            with memfile.open(**profile) as dataset:
                dataset.write(aligned_img.astype(np.uint16), 1)
            aligned_dataset = memfile.open()
            datasets_to_merge.append(aligned_dataset)
    
    # Merge all datasets (original base and aligned)
    mosaic, out_transform = merge(datasets_to_merge, method='first')
    
    # Update the base profile with the new dimensions and transform
    base_profile.update({
        'driver': 'GTiff',
        'height': mosaic.shape[1],
        'width': mosaic.shape[2],
        'transform': out_transform,
        'count': mosaic.shape[0],
        'dtype': mosaic.dtype
    })
    
    # Ensure the output directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Write the mosaic to the output file
    with rasterio.open(output_path, 'w', **base_profile) as dst:
        dst.write(mosaic)
    
    print(f"Mosaic saved at {output_path}")

# Example usage
input_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\test"
output_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\mosaic\\"
output_path = os.path.join(output_folder, "mosaic.tif")

image_paths = sorted(glob.glob(os.path.join(input_folder, "*.TIF")))
if len(image_paths) > 1:
    mosaic_images(image_paths, output_path)
else:
    print("Not enough images for mosaicking.")

WindowError: Bounds and transform are inconsistent

In [33]:
#CGT
import numpy as np
import cv2
import rasterio
import glob
import os
from rasterio.transform import Affine
from rasterio.merge import merge

def align_images(img1, img2):
    img1_scaled = cv2.normalize(img1, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    img2_scaled = cv2.normalize(img2, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1_scaled, None)
    kp2, des2 = sift.detectAndCompute(img2_scaled, None)
    
    if des1 is None or des2 is None or len(des1) < 10 or len(des2) < 10:
        raise ValueError("Feature detection failed or insufficient features found.")
    
    index_params = dict(algorithm=1, trees=5)
    search_params = dict(checks=100)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(des1, des2, k=2)
    
    good_matches = []
    for m, n in matches:
        if m.distance < 0.6 * n.distance:
            good_matches.append(m)
    
    if len(good_matches) < 10:
        raise ValueError("Not enough good matches to compute transformation.")
    
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    
    M, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 4.0)
    return M

def mosaic_images(image_paths, output_path):
    if len(image_paths) < 2:
        raise ValueError("Not enough images to create a mosaic.")
    
    raster_datasets = [rasterio.open(path) for path in image_paths]
    
    crs_list = [ds.crs for ds in raster_datasets]
    if len(set(crs_list)) > 1:
        raise ValueError("CRS mismatch between images.")
    
    base_img = raster_datasets[0].read(1)
    base_transform = raster_datasets[0].transform
    base_profile = raster_datasets[0].profile
    
    stitched_images = [base_img]
    stitched_transforms = [base_transform]
    
    for i in range(1, len(raster_datasets)):
        img = raster_datasets[i].read(1)
        try:
            M = align_images(base_img, img)
            h, w = img.shape
            aligned_img = cv2.warpPerspective(img, M, (w, h), flags=cv2.INTER_LINEAR)
            stitched_images.append(aligned_img)
        except ValueError as e:
            print(f"Skipping image {image_paths[i]} due to alignment error: {e}")
            stitched_images.append(img)
    
    mosaic, out_transform = merge(raster_datasets, method='first')
    
    print(f"Mosaic shape before writing: {mosaic.shape}")
    print(f"Output transform: {out_transform}")
    
    if mosaic.ndim == 2:
        mosaic = np.expand_dims(mosaic, axis=0)
    
    if mosaic.ndim != 3:
        raise ValueError(f"Unexpected mosaic shape: {mosaic.shape}")
    
    base_profile.update(
        dtype=rasterio.uint16,
        count=mosaic.shape[0],
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=out_transform
    )
    
    with rasterio.open(output_path, 'w', **base_profile) as dst:
        for i in range(mosaic.shape[0]):
            dst.write(mosaic[i].astype(rasterio.uint16), indexes=i+1)
    
    print(f"Mosaic saved at {output_path}")

# Example usage
input_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\test"
output_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\mosaic\\"
output_path = os.path.join(output_folder, "mosaic.tif")

image_paths = sorted(glob.glob(os.path.join(input_folder, "*.TIF")))
if len(image_paths) > 1:
    mosaic_images(image_paths, output_path)
else:
    print("Not enough images for mosaicking.")


Skipping image C:\Tarun\CIFRI_ponds\0002_MX\secondside\Red\test\yaw_corrected_georeferenced_IMG_241207_080308_0059_RED.TIF due to alignment error: Not enough good matches to compute transformation.
Skipping image C:\Tarun\CIFRI_ponds\0002_MX\secondside\Red\test\yaw_corrected_georeferenced_IMG_241207_080309_0060_RED.TIF due to alignment error: Not enough good matches to compute transformation.
Skipping image C:\Tarun\CIFRI_ponds\0002_MX\secondside\Red\test\yaw_corrected_georeferenced_IMG_241207_080311_0061_RED.TIF due to alignment error: Not enough good matches to compute transformation.
Skipping image C:\Tarun\CIFRI_ponds\0002_MX\secondside\Red\test\yaw_corrected_georeferenced_IMG_241207_080313_0062_RED.TIF due to alignment error: Not enough good matches to compute transformation.
Skipping image C:\Tarun\CIFRI_ponds\0002_MX\secondside\Red\test\yaw_corrected_georeferenced_IMG_241207_080314_0063_RED.TIF due to alignment error: Not enough good matches to compute transformation.
Skipping i

In [8]:
import numpy as np
import cv2
import rasterio
import glob
import os
from rasterio.transform import from_bounds

def align_images(img1, img2):
    img1_scaled = cv2.normalize(img1, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    img2_scaled = cv2.normalize(img2, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    import torch
    loftr = torch.hub.load('zju3dv/LoFTR', 'loftr_outdoor')
    detector = loftr  # Use pre-trained SuperGlue model
    input_dict = {'image0': torch.from_numpy(img1_scaled).float()[None, None], 'image1': torch.from_numpy(img2_scaled).float()[None, None]}
    with torch.no_grad():
        output_dict = detector(input_dict)
    matches = output_dict['matches']
    kp1, kp2 = output_dict['keypoints0'], output_dict['keypoints1']  # Get feature matches
    
    if des1 is None or des2 is None or len(des1) < 10 or len(des2) < 10:
        print('ORB failed, switching to SIFT')
        detector = cv2.SIFT_create(nfeatures=5000)
        kp1, des1 = detector.detectAndCompute(img1_scaled, None)
        kp2, des2 = detector.detectAndCompute(img2_scaled, None)  # Increase feature points
    kp1, des1 = detector.detectAndCompute(img1_scaled, None)
    kp2, des2 = detector.detectAndCompute(img2_scaled, None)
    
    if des1 is None or des2 is None or len(des1) < 10 or len(des2) < 10:
        print("Not enough keypoints detected.")
        return None  
    
    
    search_params = {}
       
    
    good_matches = matches  # SuperGlue already filters good matches
    
    if len(good_matches) < 10:
        print("Not enough good matches found.")
        return None
    
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    
    M, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
    if M is None:
        print("Homography estimation failed.")
    return M

def warp_images(image, M, output_shape):
    return cv2.warpPerspective(image, M, output_shape, flags=cv2.INTER_LINEAR)

def get_mosaic_extent(raster_datasets):
    min_x = min(ds.bounds.left for ds in raster_datasets)
    max_x = max(ds.bounds.right for ds in raster_datasets)
    min_y = min(ds.bounds.bottom for ds in raster_datasets)
    max_y = max(ds.bounds.top for ds in raster_datasets)
    return min_x, max_x, min_y, max_y

def mosaic_images(image_paths, output_path):
    if len(image_paths) < 2:
        raise ValueError("Not enough images to create a mosaic.")
    
    raster_datasets = [rasterio.open(path) for path in image_paths]
    base_img = raster_datasets[0].read(1)
    max_h, max_w = base_img.shape
    stitched_images = [base_img]
    
    for i in range(1, len(raster_datasets)):
        img = raster_datasets[i].read(1)
        M = align_images(stitched_images[-1], img)
        
        if M is not None:
            base_h, base_w = stitched_images[-1].shape
            h, w = img.shape
            if (h, w) != (base_h, base_w):
                img = cv2.resize(img, (base_w, base_h), interpolation=cv2.INTER_LINEAR)
            aligned_img = warp_images(img, M, (w * 2, h * 2))  # Expand output shape dynamically
            max_h, max_w = max(max_h, aligned_img.shape[0]), max(max_w, aligned_img.shape[1])
            canvas = np.zeros((max_h, max_w), dtype=np.uint16)
            canvas[:aligned_img.shape[0], :aligned_img.shape[1]] = aligned_img
            stitched_images.append(canvas)
            
        else:
            print(f"Skipping image {image_paths[i]} due to alignment failure.")
            padded_img = cv2.copyMakeBorder(img, 0, max(0, max_h - img.shape[0]), 0, max(0, max_w - img.shape[1]), cv2.BORDER_CONSTANT, value=0)
            max_h, max_w = max(max_h, padded_img.shape[0]), max(max_w, padded_img.shape[1])
            canvas = np.zeros((max_h, max_w), dtype=np.uint16)
            canvas[:padded_img.shape[0], :padded_img.shape[1]] = padded_img
            stitched_images.append(canvas)
            
    
    print("Stacking images with shapes:", [img.shape for img in stitched_images])
    mosaic = np.max(np.stack(stitched_images, axis=0), axis=0)
    
    min_x, max_x, min_y, max_y = get_mosaic_extent(raster_datasets)
    out_transform = from_bounds(min_x, min_y, max_x, max_y, mosaic.shape[1], mosaic.shape[0])
    
    base_profile = raster_datasets[0].profile
    base_profile.update(
        dtype=rasterio.uint16,
        count=1,
        height=mosaic.shape[0],
        width=mosaic.shape[1],
        transform=out_transform
    )
    
    with rasterio.open(output_path, 'w', **base_profile) as dst:
        dst.write(mosaic.astype(rasterio.uint16), 1)
    
    print(f"Mosaic saved at {output_path}")

# Example usage
# input_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\test"
# output_folder = "C:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\mosaic\\"
input_folder = "D:\\Tarun\\tm\\blue"
output_folder = "D:\\Tarun\\tm\\blue\\mosaic"
output_path = os.path.join(output_folder, "bluemosaic.tif")


image_paths = sorted(glob.glob(os.path.join(input_folder, "*.TIF")))
if len(image_paths) > 1:
    mosaic_images(image_paths, output_path)
else:
    print("Not enough images for mosaicking.")


C:\Users\DRONE LAB\anaconda3\envs\mxp\Lib\site-packages\torch\hub.py:293: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/zju3dv/LoFTR/zipball/master" to C:\Users\DRONE LAB/.cache\torch\hub\master.zip


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\DRONE LAB/.cache\\torch\\hub\\zju3dv_LoFTR_master\\hubconf.py'

In [7]:
import numpy as np
import cv2
import rasterio
import glob
import os
import subprocess
from rasterio.transform import from_bounds
from pyproj import Transformer

def download_dem(lat, lon, dem_folder="./dem_data/"):
    """Download DEM data for given lat/lon using NASA Earthdata or OpenTopography."""
    dem_path = os.path.join(dem_folder, f"dem_{lat}_{lon}.tif")
    if not os.path.exists(dem_path):
        print(f"Downloading DEM for {lat}, {lon}...")
        # Example: Using OpenTopography API (Requires account & credentials)
        subprocess.run(["wget", f"https://portal.opentopography.org/API/globaldem?demtype=SRTMGL1&west={lon-0.1}&east={lon+0.1}&south={lat-0.1}&north={lat+0.1}&outputFormat=GTiff", "-O", dem_path])
    return dem_path

def orthorectify_image(image_path, dem_path, output_path):
    """Use GDAL to apply DEM-based orthorectification."""
    subprocess.run([
        "gdalwarp", "-rpc", "-to", "RPC_DEM=" + dem_path,
        "-t_srs", "EPSG:4326", image_path, output_path
    ])

def align_images(img1, img2):
    """Align images using feature-based matching."""
    img1_scaled = cv2.normalize(img1, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    img2_scaled = cv2.normalize(img2, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    detector = cv2.SIFT_create(nfeatures=5000)
    kp1, des1 = detector.detectAndCompute(img1_scaled, None)
    kp2, des2 = detector.detectAndCompute(img2_scaled, None)
    
    if des1 is None or des2 is None or len(des1) < 10 or len(des2) < 10:
        print("Not enough keypoints detected.")
        return None  
    
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1, des2, k=2)
    good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
    
    if len(good_matches) < 10:
        print("Not enough good matches found.")
        return None
    
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    
    M, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
    if M is None:
        print("Homography estimation failed.")
    return M

def warp_images(image, M, output_shape):
    return cv2.warpPerspective(image, M, output_shape, flags=cv2.INTER_LINEAR)

def edge_feathering(mosaic, mask):
    """Apply edge-aware feathering using a distance transform."""
    from scipy.ndimage import distance_transform_edt
    dist = distance_transform_edt(mask)
    weight = dist / np.max(dist)
    return (mosaic * weight).astype(mosaic.dtype)

def get_mosaic_extent(raster_datasets):
    min_x = min(ds.bounds.left for ds in raster_datasets)
    max_x = max(ds.bounds.right for ds in raster_datasets)
    min_y = min(ds.bounds.bottom for ds in raster_datasets)
    max_y = max(ds.bounds.top for ds in raster_datasets)
    return min_x, max_x, min_y, max_y

def mosaic_images(image_paths, output_path):
    if len(image_paths) < 2:
        raise ValueError("Not enough images to create a mosaic.")
    
    raster_datasets = [rasterio.open(path) for path in image_paths]
    base_img = raster_datasets[0].read(1)
    max_h, max_w = base_img.shape
    stitched_images = [base_img]
    
    for i in range(1, len(raster_datasets)):
        img = raster_datasets[i].read(1)
        M = align_images(stitched_images[-1], img)
        
        if M is not None:
            base_h, base_w = stitched_images[-1].shape
            h, w = img.shape
            if (h, w) != (base_h, base_w):
                img = cv2.resize(img, (base_w, base_h), interpolation=cv2.INTER_LINEAR)
            aligned_img = warp_images(img, M, (w * 2, h * 2))
            max_h, max_w = max(max_h, aligned_img.shape[0]), max(max_w, aligned_img.shape[1])
            canvas = np.zeros((max_h, max_w), dtype=np.uint16)
            canvas[:aligned_img.shape[0], :aligned_img.shape[1]] = aligned_img
            stitched_images.append(canvas)
            
        else:
            print(f"Skipping image {image_paths[i]} due to alignment failure.")
            padded_img = cv2.copyMakeBorder(img, 0, max(0, max_h - img.shape[0]), 0, max(0, max_w - img.shape[1]), cv2.BORDER_CONSTANT, value=0)
            max_h, max_w = max(max_h, padded_img.shape[0]), max(max_w, padded_img.shape[1])
            canvas = np.zeros((max_h, max_w), dtype=np.uint16)
            canvas[:padded_img.shape[0], :padded_img.shape[1]] = padded_img
            stitched_images.append(canvas)
            
    print("Stacking images with shapes:", [img.shape for img in stitched_images])
    mosaic = np.max(np.stack(stitched_images, axis=0), axis=0)
    
    min_x, max_x, min_y, max_y = get_mosaic_extent(raster_datasets)
    out_transform = from_bounds(min_x, min_y, max_x, max_y, mosaic.shape[1], mosaic.shape[0])
    
    base_profile = raster_datasets[0].profile
    base_profile.update(
        dtype=rasterio.uint16,
        count=1,
        height=mosaic.shape[0],
        width=mosaic.shape[1],
        transform=out_transform
    )
    
    with rasterio.open(output_path, 'w', **base_profile) as dst:
        dst.write(mosaic.astype(rasterio.uint16), 1)
    
    print(f"Mosaic saved at {output_path}")

# Example usage
input_folder = "D:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\test"
output_folder = "D:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\mosaic\\"
# input_folder = "D:\\Tarun\\tm\\blue"
# output_folder = "D:\\Tarun\\tm\\blue\\mosaic"
output_path = os.path.join(output_folder, "redone.tif")


image_paths = sorted(glob.glob(os.path.join(input_folder, "*.TIF")))
if len(image_paths) > 1:
    mosaic_images(image_paths, output_path)
else:
    print("Not enough images for mosaicking.")

Not enough good matches found.
Skipping image D:\Tarun\CIFRI_ponds\0002_MX\secondside\Red\test\yaw_corrected_georeferenced_IMG_241207_080311_0061_RED.TIF due to alignment failure.
Not enough good matches found.
Skipping image D:\Tarun\CIFRI_ponds\0002_MX\secondside\Red\test\yaw_corrected_georeferenced_IMG_241207_080317_0065_RED.TIF due to alignment failure.
Stacking images with shapes: [(1304, 992), (2596, 1970), (2596, 1970), (2650, 2042), (2650, 2042), (2680, 2084), (2680, 2084), (2680, 2084), (2680, 2084), (2680, 2084)]


ValueError: all input arrays must have the same shape

In [11]:
import os
import subprocess
import numpy as np
import cv2
import rasterio
from rasterio.transform import from_origin
from pyproj import Proj, transform

def extract_metadata(image_path):
    """Extract GPS coordinates and yaw angle from image metadata using exiftool."""
    cmd = f"exiftool -GPSLatitude -GPSLongitude -XMP:Yaw {image_path}"
    result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
    lines = result.stdout.splitlines()
    gps_lat = float([line.split(":")[1].strip() for line in lines if "GPSLatitude" in line][0])
    gps_lon = float([line.split(":")[1].strip() for line in lines if "GPSLongitude" in line][0])
    yaw = float([line.split(":")[1].strip() for line in lines if "Yaw" in line][0])
    return gps_lat, gps_lon, yaw

def rotate_image(image, yaw):
    """Rotate the image based on yaw angle."""
    rows, cols = image.shape
    M = cv2.getRotationMatrix2D((cols/2, rows/2), yaw, 1)
    rotated_image = cv2.warpAffine(image, M, (cols, rows))
    return rotated_image

def georeference_image(image, gps_lat, gps_lon):
    """Georeference the image using GPS coordinates."""
    # Assuming a simple transformation for demonstration
    transform = from_origin(gps_lon - 0.01, gps_lat + 0.01, 1, 1)  # Adjust as needed
    return transform

def orthorectify_image(image_path, dem_path, output_path):
    """Apply DEM-based orthorectification using GDAL."""
    cmd = f"gdalwarp -rpc -to RPC_DEM={dem_path} -t_srs EPSG:4326 {image_path} {output_path}"
    subprocess.run(cmd, shell=True)

def mosaic_images(image_paths, output_path):
    """Mosaic multiple images into a seamless orthorectified mosaic."""
    # Read images and find the maximum dimensions
    images = [cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) for img_path in image_paths]
    max_height = max(img.shape[0] for img in images)
    max_width = max(img.shape[1] for img in images)

    # Create an empty canvas for the mosaic
    mosaic = np.zeros((max_height, max_width), dtype=np.uint16)

    # Place each image onto the mosaic canvas
    for img in images:
        h_offset = (max_height - img.shape[0]) // 2
        w_offset = (max_width - img.shape[1]) // 2
        mosaic[h_offset:h_offset+img.shape[0], w_offset:w_offset+img.shape[1]] = img

    # Save the mosaic
    cv2.imwrite(output_path, mosaic)

def main(image_folder, dem_path, output_folder):
    """Main function to process images and create a mosaic."""
    image_paths = [os.path.join(image_folder, f) for f in os.listdir(image_folder) if f.endswith('.tif')]
    georeferenced_images = []

    for image_path in image_paths:
        gps_lat, gps_lon, yaw = extract_metadata(image_path)
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        rotated_image = rotate_image(image, yaw)
        transform = georeference_image(rotated_image, gps_lat, gps_lon)
        georeferenced_images.append(rotated_image)

    # Save georeferenced images
    for i, img in enumerate(georeferenced_images):
        output_path = os.path.join(output_folder, f"georef_{i}.tif")
        cv2.imwrite(output_path, img)

    # Create mosaic
    mosaic_output_path = os.path.join(output_folder, "mosaic.tif")
    mosaic_images([os.path.join(output_folder, f"georef_{i}.tif") for i in range(len(georeferenced_images))], mosaic_output_path)

    print(f"Mosaic saved to {mosaic_output_path}")

if __name__ == "__main__":
    image_folder = "D:\\Tarun\\CIFRI_ponds\\0002_MX\\secondside\\Red\\test"
    dem_path = "path_to_your_dem.tif"
    output_folder = "path_to_output_folder"
    main(image_folder, dem_path, output_folder)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'path_to_your_images'